# DA4022: proposal-aligned final evaluation, fresh Kaggle session

Enable **Internet** and **GPU** in Kaggle settings. Use a fresh session; old cells are not required.
Run cells in order, or use Run All while monitoring the output. GPU 0 is used; two T4s do not pool VRAM.

This notebook implements controlled synthetic geometry/camera/texture/lighting variations,
attribute-derived bases, a combined basis, ordinary LoRA, random bases, real-donor bases,
and a fixed-basis + rank-1 residual hybrid. It evaluates convergence, point sparsity,
outlier robustness and unobserved-region error with donor/validation/test separation.

**Scope:** a same-day, small scene-disjoint study, not full MegaSynth replication or a large benchmark.
No positive outcome is guaranteed. All conclusions must follow the generated results.

**Expected workload:** downloads plus several thousand single-frame optimizer updates.
Actual runtime depends on Kaggle availability. A 150-minute GPU-stage wall-time cap is set below.
If the cap is reached, reporting/backup still work and completed runs are reusable.

**Saving:** every run saves CSVs and adaptive checkpoints; a ZIP is regenerated every five completed
runs and at stage boundaries. Download the latest `DA4022_FINAL_BACKUP.zip` and save the Kaggle
notebook version with outputs. Files left only in `/kaggle/working` are not durable.

**Resume after a reset:** attach the backup ZIP to the next notebook and run the optional restore
cell below BEFORE setup. Keep all scientific configuration unchanged. Completed runs are skipped;
interrupted training resumes from the most recent 10-update checkpoint.

**Reports:** `/kaggle/working/DA4022_FINAL_TODAY/report/`.
The final cell can be run manually after a failure to collect completed results.

**First execution gate:** run through the GPU preflight section first. It must print
`GPU PREFLIGHT PASSED` after all seven methods perform two updates and a saved checkpoint
is restored successfully. Then continue the remaining cells. Preflight uses validation
data and placeholder random bases only; its measurements are excluded from test tables.


## Optional restoration (leave RESTORE_ZIP empty for a fresh run)

In [ ]:
from pathlib import Path
import zipfile

RESTORE_ZIP = ""  # e.g. /kaggle/input/datasets/your-name/results/DA4022_FINAL_BACKUP.zip
if RESTORE_ZIP:
    destination = Path("/kaggle/working").resolve() if Path("/kaggle/working").exists() else (Path.cwd() / "workspace").resolve()
    with zipfile.ZipFile(RESTORE_ZIP) as archive:
        for entry in archive.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Unsafe archive path")
        archive.extractall(destination)
    print("Restored saved project. Keep the original configuration.")
else:
    print("Fresh run: no backup restored.")


## 1. Fresh-session setup and backup helpers

In [ ]:
BUILD_HASH = '2e3ddff7adc9bac8'
# Run in a fresh Kaggle notebook with Internet and GPU enabled.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from pathlib import Path
import sys, subprocess, json, hashlib, platform, shutil, time, datetime

# Support both Kaggle (/kaggle/working) and Local execution (./workspace)
BASE_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else (Path.cwd() / 'workspace').resolve()
BASE_DIR.mkdir(parents=True, exist_ok=True)
ROOT = BASE_DIR / 'DA4022_FINAL_TODAY'
ROOT.mkdir(parents=True, exist_ok=True)
REPO = BASE_DIR / 'capa'
DATA_CACHE = BASE_DIR / 'da4022_downloads'
DATA_CACHE.mkdir(exist_ok=True)

# Restore the folder DA4022_FINAL_TODAY from a previous backup here BEFORE
# running subsequent cells if resuming a new session.
def utcnow():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    temporary.replace(path)

def journal(event, **details):
    with (ROOT / 'journal.jsonl').open('a', encoding='utf-8') as f:
        f.write(json.dumps(dict(time_utc=utcnow(), event=event, **details), default=str) + '\n')

def command(args):
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with (ROOT / 'setup.log').open('a', encoding='utf-8') as f:
        f.write('\n' + repr(args) + '\n' + p.stdout)
    print(p.stdout[-2500:])
    p.check_returncode()

if not (REPO / '.git').exists():
    command(['git', 'clone', 'https://github.com/nv-dvl/capa.git', str(REPO)])

# Reuse the saved CAPA commit when resuming, rather than silently changing code.
old_env_path = ROOT / 'environment.json'
old_environment = json.loads(old_env_path.read_text()) if old_env_path.exists() else None
if old_environment and old_environment.get('capa_commit'):
    command(['git', '-C', str(REPO), 'checkout', old_environment['capa_commit']])

# Deliberately reuse Kaggle's torch/torchvision. No PEFT or TorchAO needed:
# the adaptation layers below are implemented directly in PyTorch.
command([sys.executable, '-m', 'pip', 'install', '-q',
         'huggingface_hub', 'einops', 'safetensors', 'pillow',
         'pandas', 'matplotlib', 'requests'])

import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook settings.'
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)

CAPA_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
import importlib.metadata
environment = dict(
    recorded_at=utcnow(), capa_commit=CAPA_COMMIT, python=platform.python_version(),
    packages={n: importlib.metadata.version(n) for n in
              ['torch', 'torchvision', 'numpy', 'huggingface_hub', 'einops', 'safetensors', 'pandas']},
    gpus=[dict(name=torch.cuda.get_device_name(i),
               GB=round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2))
          for i in range(torch.cuda.device_count())],
    note='Uses GPU 0 only. Two GPUs do not pool their VRAM. Determinism is best-effort; warnings are retained.')
if old_environment and old_environment['packages'] != environment['packages']:
    write_json(ROOT / ('environment_changed_' + str(int(time.time())) + '.json'), environment)
    print('WARNING: package versions differ from restored environment; inspect before mixing results.')
write_json(ROOT / 'environment.json', environment)
(ROOT / 'requirements_snapshot.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
journal('setup_complete', capa_commit=CAPA_COMMIT)
print(json.dumps(environment, indent=2))

from IPython.display import display, FileLink
def backup():
    # Excludes multi-GB downloads and the pretrained backbone, but includes
    # prepared frames, adaptive checkpoints, results, code, plots and logs.
    archive = shutil.make_archive(str(BASE_DIR / 'DA4022_FINAL_BACKUP'), 'zip',
                                  root_dir=ROOT.parent, base_dir=ROOT.name)
    print('BACKUP: download this file; /kaggle/working is not durable storage.')
    display(FileLink(archive))
    return archive

print('Setup ready. No model has been loaded yet.')

(ROOT / "code").mkdir(exist_ok=True)
(ROOT / "code" / '01_setup.py').write_text('# Run in a fresh Kaggle notebook with Internet and GPU enabled.\nimport os\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nfrom pathlib import Path\nimport sys, subprocess, json, hashlib, platform, shutil, time, datetime\n\nROOT = Path(\'/kaggle/working/DA4022_FINAL_TODAY\')\nROOT.mkdir(parents=True, exist_ok=True)\nREPO = Path(\'/kaggle/working/capa\')\nDATA_CACHE = Path(\'/kaggle/working/da4022_downloads\')\nDATA_CACHE.mkdir(exist_ok=True)\n\n# Restore the folder DA4022_FINAL_TODAY from a previous backup here BEFORE\n# running subsequent cells if resuming a new session.\ndef utcnow():\n    return datetime.datetime.now(datetime.timezone.utc).isoformat()\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, default=str), encoding=\'utf-8\')\n    temporary.replace(path)\n\ndef journal(event, **details):\n    with (ROOT / \'journal.jsonl\').open(\'a\', encoding=\'utf-8\') as f:\n        f.write(json.dumps(dict(time_utc=utcnow(), event=event, **details), default=str) + \'\\n\')\n\ndef command(args):\n    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    with (ROOT / \'setup.log\').open(\'a\', encoding=\'utf-8\') as f:\n        f.write(\'\\n\' + repr(args) + \'\\n\' + p.stdout)\n    print(p.stdout[-2500:])\n    p.check_returncode()\n\nif not (REPO / \'.git\').exists():\n    command([\'git\', \'clone\', \'https://github.com/nv-dvl/capa.git\', str(REPO)])\n\n# Reuse the saved CAPA commit when resuming, rather than silently changing code.\nold_env_path = ROOT / \'environment.json\'\nold_environment = json.loads(old_env_path.read_text()) if old_env_path.exists() else None\nif old_environment and old_environment.get(\'capa_commit\'):\n    command([\'git\', \'-C\', str(REPO), \'checkout\', old_environment[\'capa_commit\']])\n\n# Deliberately reuse Kaggle\'s torch/torchvision. No PEFT or TorchAO needed:\n# the adaptation layers below are implemented directly in PyTorch.\ncommand([sys.executable, \'-m\', \'pip\', \'install\', \'-q\',\n         \'huggingface_hub\', \'einops\', \'safetensors\', \'pillow\',\n         \'pandas\', \'matplotlib\', \'requests\'])\n\nimport torch, numpy as np, pandas as pd\nassert torch.cuda.is_available(), \'Enable GPU in Kaggle notebook settings.\'\ntorch.set_num_threads(2)\ntorch.backends.cudnn.benchmark = False\ntorch.backends.cuda.matmul.allow_tf32 = False\ntorch.backends.cudnn.allow_tf32 = False\ntorch.use_deterministic_algorithms(True, warn_only=True)\n\nCAPA_COMMIT = subprocess.check_output([\'git\', \'-C\', str(REPO), \'rev-parse\', \'HEAD\'], text=True).strip()\nimport importlib.metadata\nenvironment = dict(\n    recorded_at=utcnow(), capa_commit=CAPA_COMMIT, python=platform.python_version(),\n    packages={n: importlib.metadata.version(n) for n in\n              [\'torch\', \'torchvision\', \'numpy\', \'huggingface_hub\', \'einops\', \'safetensors\', \'pandas\']},\n    gpus=[dict(name=torch.cuda.get_device_name(i),\n               GB=round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2))\n          for i in range(torch.cuda.device_count())],\n    note=\'Uses GPU 0 only. Two GPUs do not pool their VRAM. Determinism is best-effort; warnings are retained.\')\nif old_environment and old_environment[\'packages\'] != environment[\'packages\']:\n    write_json(ROOT / (\'environment_changed_\' + str(int(time.time())) + \'.json\'), environment)\n    print(\'WARNING: package versions differ from restored environment; inspect before mixing results.\')\nwrite_json(ROOT / \'environment.json\', environment)\n(ROOT / \'requirements_snapshot.txt\').write_text(\n    subprocess.check_output([sys.executable, \'-m\', \'pip\', \'freeze\'], text=True), encoding=\'utf-8\')\njournal(\'setup_complete\', capa_commit=CAPA_COMMIT)\nprint(json.dumps(environment, indent=2))\n\nfrom IPython.display import display, FileLink\ndef backup():\n    # Excludes multi-GB downloads and the pretrained backbone, but includes\n    # prepared frames, adaptive checkpoints, results, code, plots and logs.\n    archive = shutil.make_archive(\'/kaggle/working/DA4022_FINAL_BACKUP\', \'zip\',\n                                  root_dir=ROOT.parent, base_dir=ROOT.name)\n    print(\'BACKUP: download this file; /kaggle/working is not durable storage.\')\n    display(FileLink(archive))\n    return archive\n\nprint(\'Setup ready. No model has been loaded yet.\')\n', encoding="utf-8")


## 2. Freeze the experiment configuration

In [ ]:
(ROOT / "code" / '02_config.py').write_text("# Change settings BEFORE the first experiment. Changed scientific settings\n# require a new ROOT directory; this prevents silently mixing experiments.\nCFG = {\n    'protocol_version': 'today-v1',\n    'input_width': 518,\n    'frames_per_scene': 2,\n    'donor_steps': 40,\n    'validation_steps': 30,\n    'test_steps': 100,\n    'robustness_steps': 30,\n    'rank': 4,\n    'basis_dim': 4,\n    'basis_extraction': 'robust_shared_mean_irls',\n    'irls_iterations': 5,\n    'residual_rank': 1,\n    'donor_lr': 0.001,\n    'weight_decay': 0.0,\n    'align_points': 1000,\n    'main_seeds': [47, 48],\n    'conditioning_seed': 20261122,\n    'donor_variants': ['identity'],\n    'synthetic_attributes': ['geometry', 'camera', 'texture', 'lighting'],\n    'synthetic_base_seeds': [310, 311],\n    'synthetic_levels': [-1, 0, 1],\n    'max_gpu_minutes': 150,\n    'run_robustness': True,\n    'run_appearance_ablation': True,\n    'main_conditions': {'points': 100, 'noise': 0.0},\n    'lr_candidates': {\n        'lora4': [0.0003, 0.001],\n        'lora1': [0.0003, 0.001],\n        'learned': [0.01, 0.1],\n        'learned_energy': [0.01, 0.1],\n        'random': [0.01, 0.1],\n        'hybrid': [0.01, 0.1],\n        'shared_real': [0.01, 0.1],\n    },\n    'hybrid_residual_lr': 0.001,\n    'split': {\n        'donor': ['scene0707', 'scene0766'],\n        'validation': ['scene0771'],\n        'test': ['scene0777', 'scene0806'],\n    },\n}\n# This is a small, custom single-frame adaptation protocol. Frames from a\n# sequence stay in one split, and scene means (not frame means) are reported.\n# It is NOT the original CAPA multi-frame evaluation or a large benchmark.\nscientific_cfg = {k: v for k, v in CFG.items() if k != 'max_gpu_minutes'}\nscientific_cfg['build_hash'] = globals().get('BUILD_HASH','manual-source-v1')\nCONFIG_HASH = hashlib.sha256(json.dumps(scientific_cfg, sort_keys=True).encode()).hexdigest()[:16]\nconfig_path = ROOT / 'config.json'\nif config_path.exists():\n    previous = json.loads(config_path.read_text())\n    assert previous['config_hash'] == CONFIG_HASH, (\n        'Scientific settings changed. Use a different ROOT or restore the original CFG.')\nwrite_json(config_path, {'config_hash': CONFIG_HASH, 'config': CFG, 'capa_commit': CAPA_COMMIT})\nRUNS = ROOT / 'runs'\nRUNS.mkdir(exist_ok=True)\nSESSION_START = time.monotonic()\nclass BudgetStop(RuntimeError):\n    pass\ndef check_budget():\n    if (time.monotonic() - SESSION_START) / 60 > CFG['max_gpu_minutes']:\n        raise BudgetStop('Session compute budget reached. Run the reporting and backup cells now. Resume later with the saved folder.')\nprint('Config:', CONFIG_HASH)\nprint('Donors / validation / test:', CFG['split'])\nprint('Main study:', len(CFG['main_seeds']), 'seeds; 2 test scenes; 2 frames per scene.')\n", encoding="utf-8")
# Change settings BEFORE the first experiment. Changed scientific settings
# require a new ROOT directory; this prevents silently mixing experiments.
CFG = {
    'protocol_version': 'today-v1',
    'input_width': 518,
    'frames_per_scene': 2,
    'donor_steps': 40,
    'validation_steps': 30,
    'test_steps': 100,
    'robustness_steps': 30,
    'rank': 4,
    'basis_dim': 4,
    'basis_extraction': 'robust_shared_mean_irls',
    'irls_iterations': 5,
    'residual_rank': 1,
    'donor_lr': 0.001,
    'weight_decay': 0.0,
    'align_points': 1000,
    'main_seeds': [47, 48],
    'conditioning_seed': 20261122,
    'donor_variants': ['identity'],
    'synthetic_attributes': ['geometry', 'camera', 'texture', 'lighting'],
    'synthetic_base_seeds': [310, 311],
    'synthetic_levels': [-1, 0, 1],
    'max_gpu_minutes': 150,
    'run_robustness': True,
    'run_appearance_ablation': True,
    'main_conditions': {'points': 100, 'noise': 0.0},
    'lr_candidates': {
        'lora4': [0.0003, 0.001],
        'lora1': [0.0003, 0.001],
        'learned': [0.01, 0.1],
        'learned_energy': [0.01, 0.1],
        'random': [0.01, 0.1],
        'hybrid': [0.01, 0.1],
        'shared_real': [0.01, 0.1],
    },
    'hybrid_residual_lr': 0.001,
    'split': {
        'donor': ['scene0707', 'scene0766'],
        'validation': ['scene0771'],
        'test': ['scene0777', 'scene0806'],
    },
}
# This is a small, custom single-frame adaptation protocol. Frames from a
# sequence stay in one split, and scene means (not frame means) are reported.
# It is NOT the original CAPA multi-frame evaluation or a large benchmark.
scientific_cfg = {k: v for k, v in CFG.items() if k != 'max_gpu_minutes'}
scientific_cfg['build_hash'] = globals().get('BUILD_HASH','manual-source-v1')
CONFIG_HASH = hashlib.sha256(json.dumps(scientific_cfg, sort_keys=True).encode()).hexdigest()[:16]
config_path = ROOT / 'config.json'
if config_path.exists():
    previous = json.loads(config_path.read_text())
    assert previous['config_hash'] == CONFIG_HASH, (
        'Scientific settings changed. Use a different ROOT or restore the original CFG.')
write_json(config_path, {'config_hash': CONFIG_HASH, 'config': CFG, 'capa_commit': CAPA_COMMIT})
RUNS = ROOT / 'runs'
RUNS.mkdir(exist_ok=True)
SESSION_START = time.monotonic()
class BudgetStop(RuntimeError):
    pass
def check_budget():
    if (time.monotonic() - SESSION_START) / 60 > CFG['max_gpu_minutes']:
        raise BudgetStop('Session compute budget reached. Run the reporting and backup cells now. Resume later with the saved folder.')
print('Config:', CONFIG_HASH)
print('Donors / validation / test:', CFG['split'])
print('Main study:', len(CFG['main_seeds']), 'seeds; 2 test scenes; 2 frames per scene.')


## 3. Known-optimum controlled experiment (CPU)

In [ ]:
(ROOT / "code" / '03_toy.py').write_text("# Controlled mechanism experiment. No GPU or pretrained model required.\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\n\nTOY = ROOT / 'controlled'\nTOY.mkdir(exist_ok=True)\ntoy_curves = {m: [] for m in ['Full space', 'Aligned', 'Misaligned', 'Random']}\ntoy_rows = []\nn, d, iterations, tolerance = 60, 6, 1500, 1e-6\nfor seed in range(10):\n    rng = np.random.default_rng(seed)\n    Q = np.linalg.qr(rng.normal(size=(n, n)))[0]\n    H = (Q * np.geomspace(1, 1e4, n)) @ Q.T\n    pair = np.linalg.qr(rng.normal(size=(n, 2*d)))[0]\n    U, P = pair[:, :d], pair[:, d:]\n    optimum = U @ rng.normal(size=d)\n    optimum /= np.linalg.norm(optimum)\n    bases = {'Full space': np.eye(n), 'Aligned': U,\n             'Misaligned': np.cos(np.pi/6)*U + np.sin(np.pi/6)*P,\n             'Random': np.linalg.qr(rng.normal(size=(n,d)))[0]}\n    initial = .5 * optimum @ H @ optimum\n    for method, V in bases.items():\n        h, rhs = V.T @ H @ V, V.T @ H @ optimum\n        eig = np.linalg.eigvalsh(h)\n        eta = 2 / (eig[0] + eig[-1])\n        z = np.zeros(V.shape[1])\n        exact_error = V @ np.linalg.solve(h, rhs) - optimum\n        attainable = .5 * exact_error @ H @ exact_error / initial\n        curve = []\n        for t in range(iterations+1):\n            error = V @ z - optimum\n            curve.append(max(0., float(.5 * error @ H @ error / initial)))\n            z -= eta * (h @ z - rhs)\n        toy_curves[method].append(curve)\n        reached = np.flatnonzero(np.array(curve) <= tolerance)\n        toy_rows.append(dict(seed=seed, method=method, condition_number=eig[-1]/eig[0],\n                             final_gap=curve[-1], attainable_gap=max(0., float(attainable)),\n                             steps_to_tolerance=int(reached[0]) if len(reached) else np.nan,\n                             reached=bool(len(reached))))\ntoy_df = pd.DataFrame(toy_rows)\ntoy_df.to_csv(TOY/'results.csv', index=False)\ntoy_summary = toy_df.groupby('method').agg(success_rate=('reached','mean'),\n    median_steps=('steps_to_tolerance','median'), median_final_gap=('final_gap','median'),\n    median_condition=('condition_number','median'))\ntoy_summary.to_csv(TOY/'summary.csv')\nfig, axs = plt.subplots(1,2,figsize=(12,4))\nfor method, curves in toy_curves.items():\n    a = np.array(curves)\n    for ax in axs:\n        line, = ax.plot(np.maximum(np.median(a,axis=0),1e-16), label=method)\n        ax.fill_between(np.arange(iterations+1), np.maximum(np.percentile(a,25,axis=0),1e-16),\n            np.maximum(np.percentile(a,75,axis=0),1e-16), alpha=.12,color=line.get_color())\nfor ax in axs:\n    ax.set_yscale('log'); ax.axhline(tolerance,ls=':',color='black')\n    ax.set_xlabel('Updates'); ax.set_ylabel('Loss gap / initial gap'); ax.grid(alpha=.2); ax.legend()\naxs[0].set_xlim(0,100); axs[0].set_title('Early convergence')\naxs[1].set_title('Longer convergence and irreducible error')\nfig.suptitle('Known-optimum quadratic: constructed subspaces, 10 seeds')\nfig.tight_layout()\nfor ext in ['png','pdf']:\n    fig.savefig(TOY/f'convergence.{ext}',dpi=180)\ndisplay(fig); plt.close(fig)\nprint(toy_summary)\njournal('controlled_experiment_complete', output='controlled/results.csv')\nbackup()\n", encoding="utf-8")
# Controlled mechanism experiment. No GPU or pretrained model required.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

TOY = ROOT / 'controlled'
TOY.mkdir(exist_ok=True)
toy_curves = {m: [] for m in ['Full space', 'Aligned', 'Misaligned', 'Random']}
toy_rows = []
n, d, iterations, tolerance = 60, 6, 1500, 1e-6
for seed in range(10):
    rng = np.random.default_rng(seed)
    Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
    H = (Q * np.geomspace(1, 1e4, n)) @ Q.T
    pair = np.linalg.qr(rng.normal(size=(n, 2*d)))[0]
    U, P = pair[:, :d], pair[:, d:]
    optimum = U @ rng.normal(size=d)
    optimum /= np.linalg.norm(optimum)
    bases = {'Full space': np.eye(n), 'Aligned': U,
             'Misaligned': np.cos(np.pi/6)*U + np.sin(np.pi/6)*P,
             'Random': np.linalg.qr(rng.normal(size=(n,d)))[0]}
    initial = .5 * optimum @ H @ optimum
    for method, V in bases.items():
        h, rhs = V.T @ H @ V, V.T @ H @ optimum
        eig = np.linalg.eigvalsh(h)
        eta = 2 / (eig[0] + eig[-1])
        z = np.zeros(V.shape[1])
        exact_error = V @ np.linalg.solve(h, rhs) - optimum
        attainable = .5 * exact_error @ H @ exact_error / initial
        curve = []
        for t in range(iterations+1):
            error = V @ z - optimum
            curve.append(max(0., float(.5 * error @ H @ error / initial)))
            z -= eta * (h @ z - rhs)
        toy_curves[method].append(curve)
        reached = np.flatnonzero(np.array(curve) <= tolerance)
        toy_rows.append(dict(seed=seed, method=method, condition_number=eig[-1]/eig[0],
                             final_gap=curve[-1], attainable_gap=max(0., float(attainable)),
                             steps_to_tolerance=int(reached[0]) if len(reached) else np.nan,
                             reached=bool(len(reached))))
toy_df = pd.DataFrame(toy_rows)
toy_df.to_csv(TOY/'results.csv', index=False)
toy_summary = toy_df.groupby('method').agg(success_rate=('reached','mean'),
    median_steps=('steps_to_tolerance','median'), median_final_gap=('final_gap','median'),
    median_condition=('condition_number','median'))
toy_summary.to_csv(TOY/'summary.csv')
fig, axs = plt.subplots(1,2,figsize=(12,4))
for method, curves in toy_curves.items():
    a = np.array(curves)
    for ax in axs:
        line, = ax.plot(np.maximum(np.median(a,axis=0),1e-16), label=method)
        ax.fill_between(np.arange(iterations+1), np.maximum(np.percentile(a,25,axis=0),1e-16),
            np.maximum(np.percentile(a,75,axis=0),1e-16), alpha=.12,color=line.get_color())
for ax in axs:
    ax.set_yscale('log'); ax.axhline(tolerance,ls=':',color='black')
    ax.set_xlabel('Updates'); ax.set_ylabel('Loss gap / initial gap'); ax.grid(alpha=.2); ax.legend()
axs[0].set_xlim(0,100); axs[0].set_title('Early convergence')
axs[1].set_title('Longer convergence and irreducible error')
fig.suptitle('Known-optimum quadratic: constructed subspaces, 10 seeds')
fig.tight_layout()
for ext in ['png','pdf']:
    fig.savefig(TOY/f'convergence.{ext}',dpi=180)
display(fig); plt.close(fig)
print(toy_summary)
journal('controlled_experiment_complete', output='controlled/results.csv')
backup()


## 4. Download and prepare the five official ScanNet scenes

In [ ]:
(ROOT / "code" / '04_data.py').write_text('import requests, gc\n\nSAMPLE_BASE = \'https://share.phys.ethz.ch/~pf/bingkedata/capa/sample_data/scannet_sift_noise_10pct/\'\nPREPARED = ROOT / \'data\'\nPREPARED.mkdir(exist_ok=True)\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef download_sample(scene):\n    # Reuse an attached official sample when available.\n    attached = sorted(Path(\'/kaggle/input\').rglob(scene + \'.pt\'))\n    if attached:\n        return attached[0]\n    target = DATA_CACHE / (scene + \'.pt\')\n    if target.exists():\n        return target\n    part = target.with_suffix(\'.pt.part\')\n    url = SAMPLE_BASE + scene + \'.pt\'\n    for attempt in range(2):\n        try:\n            offset = part.stat().st_size if part.exists() else 0\n            headers = {\'Range\': f\'bytes={offset}-\'} if offset else {}\n            with requests.get(url, headers=headers, stream=True, timeout=(25,90)) as response:\n                response.raise_for_status()\n                append = offset > 0 and response.status_code == 206\n                total = response.headers.get(\'Content-Length\',\'unknown\')\n                print(\'Downloading\', scene, \'response bytes:\', total, flush=True)\n                with part.open(\'ab\' if append else \'wb\') as f:\n                    for chunk in response.iter_content(1024*1024):\n                        if chunk:\n                            f.write(chunk)\n            part.replace(target)\n            return target\n        except Exception as exc:\n            journal(\'download_error\', scene=scene, error=str(exc), attempt=attempt)\n            if attempt == 1:\n                raise RuntimeError(f\'Could not download {scene}. Attach the official .pt file as a Kaggle input and rerun this cell.\') from exc\n\nmanifest_path = ROOT / \'data_manifest.json\'\nif manifest_path.exists():\n    MANIFEST = json.loads(manifest_path.read_text())\n    assert MANIFEST[\'config_hash\'] == CONFIG_HASH\n    for record in MANIFEST[\'frames\']:\n        file = ROOT / record[\'file\']\n        assert file.exists() and sha256(file) == record[\'sha256\'], f\'Missing/changed prepared frame: {file}\'\n    print(\'Reusing verified prepared frames.\')\nelse:\n    records, sources = [], []\n    for role, scenes in CFG[\'split\'].items():\n        for scene in scenes:\n            source = download_sample(scene)\n            # Official/public CAPA tensors or the user\'s attached copies only.\n            pack = torch.load(source, map_location=\'cpu\', weights_only=False)\n            assert \'depth_gt_nvhw\' in pack, f\'{scene} has no dense GT; cannot use for this study.\'\n            rgb = pack[\'rgb_nv3hw\'].float()\n            truth = pack[\'depth_gt_nvhw\'].float()\n            H,W = rgb.shape[-2:]\n            rgb = rgb.reshape(-1,3,H,W)\n            truth = truth.reshape(-1,H,W)\n            assert len(rgb) == len(truth)\n            indices = np.unique(np.linspace(0,len(rgb)-1,CFG[\'frames_per_scene\'],dtype=int))\n            source_hash = sha256(source)\n            sources.append(dict(scene=scene, role=role, url=SAMPLE_BASE+scene+\'.pt\',\n                                sha256=source_hash, total_frames=len(rgb), selected_frames=indices.tolist()))\n            for idx in indices:\n                image = rgb[int(idx)].numpy()\n                if np.nanmax(image) > 1.5:\n                    image = image / 255.\n                assert np.isfinite(image).all() and image.min() >= 0 and image.max() <= 1.01\n                gt = truth[int(idx)].numpy()\n                valid = np.isfinite(gt) & (gt > 0)\n                assert valid.sum() >= 1000, \'Insufficient valid depth.\'\n                # Preserve validity explicitly; no interpolation of GT or masks.\n                gt = np.where(valid,gt,0).astype(\'float32\')\n                sample_id = f\'{scene}_frame{int(idx):04d}\'\n                file = PREPARED / (sample_id+\'.npz\')\n                np.savez_compressed(file, rgb=np.clip(image,0,1).astype(\'float32\'), gt=gt, valid=valid)\n                records.append(dict(id=sample_id,scene=scene,role=role,\n                                    frame_index=int(idx),file=str(file.relative_to(ROOT)),sha256=sha256(file)))\n            del pack,rgb,truth\n            gc.collect()\n            print(\'Prepared\', role, scene, \'frames\', indices.tolist(), flush=True)\n    assert len({x[\'scene\'] for x in records}) == sum(map(len,CFG[\'split\'].values()))\n    MANIFEST = dict(config_hash=CONFIG_HASH, protocol=\'single-frame adaptation; regenerated synthetic sparse depth\',\n                    frames=records,sources=sources)\n    write_json(manifest_path, MANIFEST)\n\ndef records_for(role):\n    return [r for r in MANIFEST[\'frames\'] if r[\'role\']==role]\n\ndef load_frame(record, variant=\'identity\'):\n    with np.load(ROOT / record[\'file\']) as a:\n        rgb,gt,valid = a[\'rgb\'].copy(),a[\'gt\'].copy(),a[\'valid\'].copy()\n    if variant==\'dark\':\n        rgb = rgb * .6\n    elif variant==\'bright\':\n        rgb = np.clip(rgb * 1.4,0,1)\n    elif variant!=\'identity\':\n        raise ValueError(variant)\n    return torch.from_numpy(rgb[None]), torch.from_numpy(gt[None]), torch.from_numpy(valid[None])\n\ndef make_condition(record, points=100, noise=0., variant=\'identity\'):\n    rgb,gt,valid = load_frame(record,variant)\n    # Same locations/noise across methods and training seeds. For different\n    # point budgets the locations are nested by prefix of one permutation.\n    key = f\'{CFG["conditioning_seed"]}/{record["id"]}\'\n    seed = int(hashlib.sha256(key.encode()).hexdigest()[:8],16)\n    rng = np.random.default_rng(seed)\n    eligible = np.flatnonzero(valid.numpy().reshape(-1))\n    indices = rng.permutation(eligible)[:points]\n    assert len(indices) == points\n    mask = torch.zeros_like(valid).reshape(-1)\n    mask[torch.from_numpy(indices.copy())] = True\n    mask = mask.reshape_as(valid)\n    condition = torch.where(mask,gt,torch.zeros_like(gt))\n    noise_rng = np.random.default_rng(seed ^ 0xA5A5A5A5)\n    k = int(round(points*noise))\n    selected = noise_rng.permutation(points)[:k]\n    factors = noise_rng.uniform(.2,2.,size=points).astype(\'float32\')\n    corrupted = indices[selected]\n    if k:\n        flat = condition.reshape(-1)\n        flat[torch.from_numpy(corrupted.copy())] *= torch.from_numpy(factors[selected].copy())\n    return rgb,gt,valid,condition,mask,dict(points=points,noise_fraction=noise,\n        actual_corrupted_points=k,seed=seed,noise_model=\'multiplicative replacement factor uniform[0.2,2.0]\')\n\nfig,axes = plt.subplots(len(CFG[\'split\']),4,figsize=(12,8),squeeze=False)\nfor row,role in enumerate(CFG[\'split\']):\n    for ax in axes[row]: ax.axis(\'off\')\n    for ax,rec in zip(axes[row],records_for(role)):\n        image,_,_ = load_frame(rec)\n        ax.imshow(image[0].permute(1,2,0)); ax.set_title(role+\'\\n\'+rec[\'id\'],fontsize=8)\nfig.suptitle(\'Fixed splits: no sequence crosses donor / validation / test\')\nfig.tight_layout(); fig.savefig(ROOT/\'split_contact_sheet.png\',dpi=150)\ndisplay(fig); plt.close(fig)\nprint(pd.DataFrame(MANIFEST[\'frames\'])[[\'id\',\'scene\',\'role\']].to_string(index=False))\njournal(\'data_ready\',frames=len(MANIFEST[\'frames\']),scenes=len(MANIFEST[\'sources\']))\nbackup()\n', encoding="utf-8")
import requests, gc

SAMPLE_BASE = 'https://share.phys.ethz.ch/~pf/bingkedata/capa/sample_data/scannet_sift_noise_10pct/'
PREPARED = ROOT / 'data'
PREPARED.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def download_sample(scene):
    # Reuse an attached official sample when available.
    attached = sorted(Path('/kaggle/input').rglob(scene + '.pt'))
    if attached:
        return attached[0]
    target = DATA_CACHE / (scene + '.pt')
    if target.exists():
        return target
    part = target.with_suffix('.pt.part')
    url = SAMPLE_BASE + scene + '.pt'
    for attempt in range(2):
        try:
            offset = part.stat().st_size if part.exists() else 0
            headers = {'Range': f'bytes={offset}-'} if offset else {}
            with requests.get(url, headers=headers, stream=True, timeout=(25,90)) as response:
                response.raise_for_status()
                append = offset > 0 and response.status_code == 206
                total = response.headers.get('Content-Length','unknown')
                print('Downloading', scene, 'response bytes:', total, flush=True)
                with part.open('ab' if append else 'wb') as f:
                    for chunk in response.iter_content(1024*1024):
                        if chunk:
                            f.write(chunk)
            part.replace(target)
            return target
        except Exception as exc:
            journal('download_error', scene=scene, error=str(exc), attempt=attempt)
            if attempt == 1:
                raise RuntimeError(f'Could not download {scene}. Attach the official .pt file as a Kaggle input and rerun this cell.') from exc

manifest_path = ROOT / 'data_manifest.json'
if manifest_path.exists():
    MANIFEST = json.loads(manifest_path.read_text())
    assert MANIFEST['config_hash'] == CONFIG_HASH
    for record in MANIFEST['frames']:
        file = ROOT / record['file']
        assert file.exists() and sha256(file) == record['sha256'], f'Missing/changed prepared frame: {file}'
    print('Reusing verified prepared frames.')
else:
    records, sources = [], []
    for role, scenes in CFG['split'].items():
        for scene in scenes:
            source = download_sample(scene)
            # Official/public CAPA tensors or the user's attached copies only.
            pack = torch.load(source, map_location='cpu', weights_only=False)
            assert 'depth_gt_nvhw' in pack, f'{scene} has no dense GT; cannot use for this study.'
            rgb = pack['rgb_nv3hw'].float()
            truth = pack['depth_gt_nvhw'].float()
            H,W = rgb.shape[-2:]
            rgb = rgb.reshape(-1,3,H,W)
            truth = truth.reshape(-1,H,W)
            assert len(rgb) == len(truth)
            indices = np.unique(np.linspace(0,len(rgb)-1,CFG['frames_per_scene'],dtype=int))
            source_hash = sha256(source)
            sources.append(dict(scene=scene, role=role, url=SAMPLE_BASE+scene+'.pt',
                                sha256=source_hash, total_frames=len(rgb), selected_frames=indices.tolist()))
            for idx in indices:
                image = rgb[int(idx)].numpy()
                if np.nanmax(image) > 1.5:
                    image = image / 255.
                assert np.isfinite(image).all() and image.min() >= 0 and image.max() <= 1.01
                gt = truth[int(idx)].numpy()
                valid = np.isfinite(gt) & (gt > 0)
                assert valid.sum() >= 1000, 'Insufficient valid depth.'
                # Preserve validity explicitly; no interpolation of GT or masks.
                gt = np.where(valid,gt,0).astype('float32')
                sample_id = f'{scene}_frame{int(idx):04d}'
                file = PREPARED / (sample_id+'.npz')
                np.savez_compressed(file, rgb=np.clip(image,0,1).astype('float32'), gt=gt, valid=valid)
                records.append(dict(id=sample_id,scene=scene,role=role,
                                    frame_index=int(idx),file=str(file.relative_to(ROOT)),sha256=sha256(file)))
            del pack,rgb,truth
            gc.collect()
            print('Prepared', role, scene, 'frames', indices.tolist(), flush=True)
    assert len({x['scene'] for x in records}) == sum(map(len,CFG['split'].values()))
    MANIFEST = dict(config_hash=CONFIG_HASH, protocol='single-frame adaptation; regenerated synthetic sparse depth',
                    frames=records,sources=sources)
    write_json(manifest_path, MANIFEST)

def records_for(role):
    return [r for r in MANIFEST['frames'] if r['role']==role]

def load_frame(record, variant='identity'):
    with np.load(ROOT / record['file']) as a:
        rgb,gt,valid = a['rgb'].copy(),a['gt'].copy(),a['valid'].copy()
    if variant=='dark':
        rgb = rgb * .6
    elif variant=='bright':
        rgb = np.clip(rgb * 1.4,0,1)
    elif variant!='identity':
        raise ValueError(variant)
    return torch.from_numpy(rgb[None]), torch.from_numpy(gt[None]), torch.from_numpy(valid[None])

def make_condition(record, points=100, noise=0., variant='identity'):
    rgb,gt,valid = load_frame(record,variant)
    # Same locations/noise across methods and training seeds. For different
    # point budgets the locations are nested by prefix of one permutation.
    key = f'{CFG["conditioning_seed"]}/{record["id"]}'
    seed = int(hashlib.sha256(key.encode()).hexdigest()[:8],16)
    rng = np.random.default_rng(seed)
    eligible = np.flatnonzero(valid.numpy().reshape(-1))
    indices = rng.permutation(eligible)[:points]
    assert len(indices) == points
    mask = torch.zeros_like(valid).reshape(-1)
    mask[torch.from_numpy(indices.copy())] = True
    mask = mask.reshape_as(valid)
    condition = torch.where(mask,gt,torch.zeros_like(gt))
    noise_rng = np.random.default_rng(seed ^ 0xA5A5A5A5)
    k = int(round(points*noise))
    selected = noise_rng.permutation(points)[:k]
    factors = noise_rng.uniform(.2,2.,size=points).astype('float32')
    corrupted = indices[selected]
    if k:
        flat = condition.reshape(-1)
        flat[torch.from_numpy(corrupted.copy())] *= torch.from_numpy(factors[selected].copy())
    return rgb,gt,valid,condition,mask,dict(points=points,noise_fraction=noise,
        actual_corrupted_points=k,seed=seed,noise_model='multiplicative replacement factor uniform[0.2,2.0]')

fig,axes = plt.subplots(len(CFG['split']),4,figsize=(12,8),squeeze=False)
for row,role in enumerate(CFG['split']):
    for ax in axes[row]: ax.axis('off')
    for ax,rec in zip(axes[row],records_for(role)):
        image,_,_ = load_frame(rec)
        ax.imshow(image[0].permute(1,2,0)); ax.set_title(role+'\n'+rec['id'],fontsize=8)
fig.suptitle('Fixed splits: no sequence crosses donor / validation / test')
fig.tight_layout(); fig.savefig(ROOT/'split_contact_sheet.png',dpi=150)
display(fig); plt.close(fig)
print(pd.DataFrame(MANIFEST['frames'])[['id','scene','role']].to_string(index=False))
journal('data_ready',frames=len(MANIFEST['frames']),scenes=len(MANIFEST['sources']))
backup()


## 5. Controlled geometry, camera, texture and lighting data

In [ ]:
(ROOT / "code" / '05_synthetic.py').write_text('# A small controlled renderer supplies actual scene/camera/material/light\n# interventions. It is intentionally simple, not MegaSynth or photorealistic.\nSYNTH = ROOT / \'synthetic\'\nSYNTH.mkdir(exist_ok=True)\n\ndef render_controlled(seed, attribute, level, height=240, width=320):\n    assert attribute in CFG[\'synthetic_attributes\']\n    rng = np.random.default_rng(seed)\n    # Base scene parameters are identical across attributes for a given seed.\n    colors = rng.uniform(.25,.85,size=(7,3)).astype(np.float32)\n    center = np.array([-.7 + rng.uniform(-.2,.2), -.65, 3.4],dtype=np.float32)\n    radius = .65\n    box_center = np.array([.85,-.8,4.1],dtype=np.float32)\n    box_half = np.array([.55,.65,.55],dtype=np.float32)\n    camera = np.array([0.,0.,0.],dtype=np.float32)\n    target = np.array([0.,-.1,4.],dtype=np.float32)\n    light = np.array([-1.,2.,-2.],dtype=np.float32)\n    texture_frequency, texture_contrast = 2.0, .22\n    if attribute==\'geometry\':\n        radius *= 1 + .25*level\n        center[0] += .25*level\n        box_half[1] *= 1 + .25*level\n    elif attribute==\'camera\':\n        camera[0] += .35*level\n        camera[1] += .12*level\n    elif attribute==\'texture\':\n        texture_frequency = 2.0 ** (level+1)\n    elif attribute==\'lighting\':\n        light[0] += 2.2*level\n        light[2] += .5*level\n    forward = target-camera; forward /= np.linalg.norm(forward)\n    right = np.cross(forward,np.array([0.,1.,0.])); right /= np.linalg.norm(right)\n    # For forward +Z, cross(forward,up) is -X; use a consistent right-handed camera.\n    right = -right\n    up = np.cross(forward,right); up /= np.linalg.norm(up)\n    yy,xx = np.mgrid[0:height,0:width]\n    u = (2*(xx+.5)/width-1)*np.tan(np.deg2rad(60)/2)\n    v = (1-2*(yy+.5)/height)*np.tan(np.deg2rad(60)/2)*(height/width)\n    rays = forward + u[...,None]*right + v[...,None]*up\n    depth = np.full((height,width),np.inf)\n    normal = np.zeros((height,width,3))\n    material = np.zeros((height,width),dtype=np.int32)\n    def install(t,n,index):\n        take = (t>1e-3)&(t<depth)\n        depth[take]=t[take]\n        normal[take]=np.broadcast_to(n,normal.shape)[take]\n        material[take]=index\n    # Back wall, floor, ceiling, and side walls.\n    planes = [(2,6.,[0,0,-1],0),(1,-1.5,[0,1,0],1),\n              (1,2.,[0,-1,0],2),(0,-3.,[1,0,0],3),(0,3.,[-1,0,0],4)]\n    for axis,value,n,index in planes:\n        denominator = rays[...,axis]\n        denominator = np.where(np.abs(denominator)<1e-8,1e-8,denominator)\n        install((value-camera[axis])/denominator,np.array(n),index)\n    # Sphere intersection, using unnormalised rays (t is camera-Z depth).\n    oc = camera-center\n    aa = (rays*rays).sum(-1)\n    bb = 2*(rays*oc).sum(-1)\n    cc = (oc*oc).sum()-radius**2\n    discriminant = bb**2-4*aa*cc\n    t = (-bb-np.sqrt(np.maximum(discriminant,0)))/(2*aa)\n    t = np.where(discriminant>0,t,np.inf)\n    finite_t = np.where(np.isfinite(t),t,0)\n    normals = (camera + finite_t[...,None]*rays-center)/radius\n    install(t,normals,5)\n    # Axis-aligned box using slab intersections.\n    safe_rays = np.where(np.abs(rays)<1e-8,1e-8,rays)\n    t0 = (box_center-box_half-camera)/safe_rays\n    t1 = (box_center+box_half-camera)/safe_rays\n    enter = np.minimum(t0,t1).max(-1)\n    leave = np.maximum(t0,t1).min(-1)\n    box_t = np.where((leave>=enter)&(enter>0),enter,np.inf)\n    p = camera + np.where(np.isfinite(box_t),box_t,0)[...,None]*rays\n    local = (p-box_center)/box_half\n    axis = np.abs(local).argmax(-1)\n    box_n = np.zeros_like(rays)\n    for j in range(3):\n        box_n[...,j] = np.where(axis==j,np.sign(local[...,j]),0)\n    install(box_t,box_n,6)\n    points = camera+depth[...,None]*rays\n    pattern = np.sin(texture_frequency*points[...,0]*np.pi)*np.sin(texture_frequency*points[...,2]*np.pi)\n    albedo = colors[material]*(1+texture_contrast*pattern[...,None])\n    light /= np.linalg.norm(light)\n    lambert = np.maximum((normal*light).sum(-1),0)\n    rgb = np.clip(albedo*(.3+.7*lambert[...,None]),0,1)\n    valid = np.isfinite(depth)&(depth>0)&(depth<50)\n    return rgb.transpose(2,0,1).astype(\'float32\'),np.where(valid,depth,0).astype(\'float32\'),valid\n\nSYNTH_TASKS = []\nfor attribute in CFG[\'synthetic_attributes\']:\n    for seed in CFG[\'synthetic_base_seeds\']:\n        task = []\n        for level in CFG[\'synthetic_levels\']:\n            sample_id = f\'synth_{attribute}_{seed}_{level:+d}\'\n            file = PREPARED/(sample_id+\'.npz\')\n            rgb,gt,valid = render_controlled(seed,attribute,level)\n            np.savez_compressed(file,rgb=rgb,gt=gt,valid=valid)\n            task.append(dict(id=sample_id,scene=f\'synthetic_base_{seed}\',role=\'synthetic_donor\',\n                attribute=attribute,level=level,file=str(file.relative_to(ROOT)),sha256=sha256(file)))\n        SYNTH_TASKS.append(dict(id=f\'{attribute}_{seed}\',attribute=attribute,frames=task))\nwrite_json(ROOT/\'synthetic_manifest.json\',dict(tasks=SYNTH_TASKS,\n    description=\'Analytic room/box/sphere ray casting; Lambertian shading; controlled target intervention; nuisance scene seed fixed within each task.\'))\nfig,axs=plt.subplots(4,3,figsize=(10,10))\nfor row,attribute in enumerate(CFG[\'synthetic_attributes\']):\n    task=next(t for t in SYNTH_TASKS if t[\'attribute\']==attribute)\n    for ax,rec in zip(axs[row],task[\'frames\']):\n        rgb,_,_=load_frame(rec)\n        ax.imshow(rgb[0].permute(1,2,0));ax.set_title(attribute+f\' level {rec["level"]}\');ax.axis(\'off\')\nfig.tight_layout();fig.savefig(SYNTH/\'controlled_variations.png\',dpi=160)\ndisplay(fig);plt.close(fig)\njournal(\'synthetic_data_ready\',tasks=len(SYNTH_TASKS),attributes=CFG[\'synthetic_attributes\'])\n', encoding="utf-8")
# A small controlled renderer supplies actual scene/camera/material/light
# interventions. It is intentionally simple, not MegaSynth or photorealistic.
SYNTH = ROOT / 'synthetic'
SYNTH.mkdir(exist_ok=True)

def render_controlled(seed, attribute, level, height=240, width=320):
    assert attribute in CFG['synthetic_attributes']
    rng = np.random.default_rng(seed)
    # Base scene parameters are identical across attributes for a given seed.
    colors = rng.uniform(.25,.85,size=(7,3)).astype(np.float32)
    center = np.array([-.7 + rng.uniform(-.2,.2), -.65, 3.4],dtype=np.float32)
    radius = .65
    box_center = np.array([.85,-.8,4.1],dtype=np.float32)
    box_half = np.array([.55,.65,.55],dtype=np.float32)
    camera = np.array([0.,0.,0.],dtype=np.float32)
    target = np.array([0.,-.1,4.],dtype=np.float32)
    light = np.array([-1.,2.,-2.],dtype=np.float32)
    texture_frequency, texture_contrast = 2.0, .22
    if attribute=='geometry':
        radius *= 1 + .25*level
        center[0] += .25*level
        box_half[1] *= 1 + .25*level
    elif attribute=='camera':
        camera[0] += .35*level
        camera[1] += .12*level
    elif attribute=='texture':
        texture_frequency = 2.0 ** (level+1)
    elif attribute=='lighting':
        light[0] += 2.2*level
        light[2] += .5*level
    forward = target-camera; forward /= np.linalg.norm(forward)
    right = np.cross(forward,np.array([0.,1.,0.])); right /= np.linalg.norm(right)
    # For forward +Z, cross(forward,up) is -X; use a consistent right-handed camera.
    right = -right
    up = np.cross(forward,right); up /= np.linalg.norm(up)
    yy,xx = np.mgrid[0:height,0:width]
    u = (2*(xx+.5)/width-1)*np.tan(np.deg2rad(60)/2)
    v = (1-2*(yy+.5)/height)*np.tan(np.deg2rad(60)/2)*(height/width)
    rays = forward + u[...,None]*right + v[...,None]*up
    depth = np.full((height,width),np.inf)
    normal = np.zeros((height,width,3))
    material = np.zeros((height,width),dtype=np.int32)
    def install(t,n,index):
        take = (t>1e-3)&(t<depth)
        depth[take]=t[take]
        normal[take]=np.broadcast_to(n,normal.shape)[take]
        material[take]=index
    # Back wall, floor, ceiling, and side walls.
    planes = [(2,6.,[0,0,-1],0),(1,-1.5,[0,1,0],1),
              (1,2.,[0,-1,0],2),(0,-3.,[1,0,0],3),(0,3.,[-1,0,0],4)]
    for axis,value,n,index in planes:
        denominator = rays[...,axis]
        denominator = np.where(np.abs(denominator)<1e-8,1e-8,denominator)
        install((value-camera[axis])/denominator,np.array(n),index)
    # Sphere intersection, using unnormalised rays (t is camera-Z depth).
    oc = camera-center
    aa = (rays*rays).sum(-1)
    bb = 2*(rays*oc).sum(-1)
    cc = (oc*oc).sum()-radius**2
    discriminant = bb**2-4*aa*cc
    t = (-bb-np.sqrt(np.maximum(discriminant,0)))/(2*aa)
    t = np.where(discriminant>0,t,np.inf)
    finite_t = np.where(np.isfinite(t),t,0)
    normals = (camera + finite_t[...,None]*rays-center)/radius
    install(t,normals,5)
    # Axis-aligned box using slab intersections.
    safe_rays = np.where(np.abs(rays)<1e-8,1e-8,rays)
    t0 = (box_center-box_half-camera)/safe_rays
    t1 = (box_center+box_half-camera)/safe_rays
    enter = np.minimum(t0,t1).max(-1)
    leave = np.maximum(t0,t1).min(-1)
    box_t = np.where((leave>=enter)&(enter>0),enter,np.inf)
    p = camera + np.where(np.isfinite(box_t),box_t,0)[...,None]*rays
    local = (p-box_center)/box_half
    axis = np.abs(local).argmax(-1)
    box_n = np.zeros_like(rays)
    for j in range(3):
        box_n[...,j] = np.where(axis==j,np.sign(local[...,j]),0)
    install(box_t,box_n,6)
    points = camera+depth[...,None]*rays
    pattern = np.sin(texture_frequency*points[...,0]*np.pi)*np.sin(texture_frequency*points[...,2]*np.pi)
    albedo = colors[material]*(1+texture_contrast*pattern[...,None])
    light /= np.linalg.norm(light)
    lambert = np.maximum((normal*light).sum(-1),0)
    rgb = np.clip(albedo*(.3+.7*lambert[...,None]),0,1)
    valid = np.isfinite(depth)&(depth>0)&(depth<50)
    return rgb.transpose(2,0,1).astype('float32'),np.where(valid,depth,0).astype('float32'),valid

SYNTH_TASKS = []
for attribute in CFG['synthetic_attributes']:
    for seed in CFG['synthetic_base_seeds']:
        task = []
        for level in CFG['synthetic_levels']:
            sample_id = f'synth_{attribute}_{seed}_{level:+d}'
            file = PREPARED/(sample_id+'.npz')
            rgb,gt,valid = render_controlled(seed,attribute,level)
            np.savez_compressed(file,rgb=rgb,gt=gt,valid=valid)
            task.append(dict(id=sample_id,scene=f'synthetic_base_{seed}',role='synthetic_donor',
                attribute=attribute,level=level,file=str(file.relative_to(ROOT)),sha256=sha256(file)))
        SYNTH_TASKS.append(dict(id=f'{attribute}_{seed}',attribute=attribute,frames=task))
write_json(ROOT/'synthetic_manifest.json',dict(tasks=SYNTH_TASKS,
    description='Analytic room/box/sphere ray casting; Lambertian shading; controlled target intervention; nuisance scene seed fixed within each task.'))
fig,axs=plt.subplots(4,3,figsize=(10,10))
for row,attribute in enumerate(CFG['synthetic_attributes']):
    task=next(t for t in SYNTH_TASKS if t['attribute']==attribute)
    for ax,rec in zip(axs[row],task['frames']):
        rgb,_,_=load_frame(rec)
        ax.imshow(rgb[0].permute(1,2,0));ax.set_title(attribute+f' level {rec["level"]}');ax.axis('off')
fig.tight_layout();fig.savefig(SYNTH/'controlled_variations.png',dpi=160)
display(fig);plt.close(fig)
journal('synthetic_data_ready',tasks=len(SYNTH_TASKS),attributes=CFG['synthetic_attributes'])


## 6. Load VGGT and verify the zero-update baseline

In [ ]:
(ROOT / "code" / '06_engine.py').write_text('import random, math, traceback, importlib.util\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n# Import VGGT directly. This avoids CAPA\'s package/PEFT/TorchAO import chain.\nsys.path.insert(0,str(REPO/\'third_party\'/\'VGGT_VPT\'))\nfrom vggt.models.vggt import VGGT\nspec = importlib.util.spec_from_file_location(\'da4022_alignment\',REPO/\'capa\'/\'utils\'/\'alignment.py\')\nalignment = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(alignment)\n\ndef seed_everything(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\ndef float_tree(x):\n    if torch.is_tensor(x): return x.float()\n    if isinstance(x,list): return [float_tree(v) for v in x]\n    if isinstance(x,tuple): return tuple(float_tree(v) for v in x)\n    return x\n\ndef checkpoint_forward(original):\n    def forward(*args,**kwargs):\n        if torch.is_grad_enabled():\n            return checkpoint(original,*args,use_reentrant=False,**kwargs)\n        return original(*args,**kwargs)\n    return forward\n\nclass AdaptedLinear(nn.Module):\n    def __init__(self,base,method,rank=4,bases=None,residual_rank=1):\n        super().__init__()\n        self.base=base; self.method=method\n        dev=base.weight.device\n        out_features,in_features=base.weight.shape\n        if method in [\'lora4\',\'lora1\',\'donor\']:\n            self.A=nn.Parameter(torch.empty(rank,in_features,device=dev,dtype=torch.float32))\n            self.B=nn.Parameter(torch.zeros(out_features,rank,device=dev,dtype=torch.float32))\n            nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))\n        else:\n            U,V=bases\n            self.register_buffer(\'U\',U.to(dev).float())\n            self.register_buffer(\'V\',V.to(dev).float())\n            self.M=nn.Parameter(torch.zeros(U.shape[1],V.shape[1],device=dev,dtype=torch.float32))\n            if method==\'hybrid\':\n                self.A=nn.Parameter(torch.empty(residual_rank,in_features,device=dev,dtype=torch.float32))\n                self.B=nn.Parameter(torch.zeros(out_features,residual_rank,device=dev,dtype=torch.float32))\n                nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))\n        self.base.requires_grad_(False)\n\n    def forward(self,x):\n        y=self.base(x)\n        if hasattr(self,\'M\'):\n            y=y+2*F.linear(F.linear(F.linear(x,self.V.T),self.M),self.U)\n        if hasattr(self,\'A\'):\n            y=y+2*F.linear(F.linear(x,self.A),self.B)\n        return y\n\nclass Runner:\n    def __init__(self):\n        self.device=torch.device(\'cuda:0\')\n        self.dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        print(\'Loading VGGT checkpoint. First download can take several minutes.\',flush=True)\n        self.model=VGGT.from_pretrained(\'facebook/VGGT-1B\').eval().requires_grad_(False).to(self.device)\n        self.names=[name for name,layer in self.model.named_modules()\n                    if isinstance(layer,nn.Linear) and \'aggregator.patch_embed.\' in name and name.endswith(\'.qkv\')]\n        assert len(self.names)==24, f\'Unexpected VGGT layout: {len(self.names)} image-encoder QKV layers.\'\n        for block in self.model.modules():\n            if block.__class__.__name__ in {\'Block\',\'NestedTensorBlock\'}:\n                block.forward=checkpoint_forward(block.forward)\n        self.adapters={}\n        print(\'Loaded. Precision:\',self.dtype,\'target layers:\',len(self.names),flush=True)\n\n    def clear(self):\n        for name,layer in list(self.adapters.items()):\n            parent,child=name.rsplit(\'.\',1)\n            setattr(self.model.get_submodule(parent),child,layer.base)\n        self.adapters={}\n        self.model.requires_grad_(False)\n        gc.collect();torch.cuda.empty_cache()\n\n    def install(self,method,seed,banks=None):\n        self.clear();seed_everything(seed)\n        if method==\'frozen\':return\n        for i,name in enumerate(self.names):\n            base=self.model.get_submodule(name)\n            basis=None\n            rank=1 if method==\'lora1\' else CFG[\'rank\']\n            if method==\'random\':\n                g=torch.Generator().manual_seed(seed+76+i*1009)\n                U=torch.linalg.qr(torch.randn(base.out_features,CFG[\'basis_dim\'],generator=g),mode=\'reduced\').Q\n                V=torch.linalg.qr(torch.randn(base.in_features,CFG[\'basis_dim\'],generator=g),mode=\'reduced\').Q\n                basis=(U,V)\n            elif method not in [\'donor\',\'lora4\',\'lora1\']:\n                assert banks is not None and name in banks, f\'Missing bases for {name}\'\n                basis=(banks[name][\'U\'],banks[name][\'V\'])\n            layer=AdaptedLinear(base,method,rank,basis,CFG[\'residual_rank\'])\n            parent,child=name.rsplit(\'.\',1)\n            setattr(self.model.get_submodule(parent),child,layer)\n            self.adapters[name]=layer\n        self.model.eval()\n\n    def groups(self,lr):\n        groups=[]\n        for layer in self.adapters.values():\n            if hasattr(layer,\'M\'):groups.append({\'params\':[layer.M],\'lr\':lr})\n            if hasattr(layer,\'A\'):\n                rate=CFG[\'hybrid_residual_lr\'] if layer.method==\'hybrid\' else lr\n                groups.append({\'params\':[layer.A,layer.B],\'lr\':rate})\n        return groups\n\n    def adapter_state(self):\n        return {name:{key:p.detach().cpu().clone() for key,p in layer.named_parameters(recurse=False)}\n                for name,layer in self.adapters.items()}\n\n    def restore_adapter(self,state):\n        assert set(state)==set(self.adapters)\n        with torch.no_grad():\n            for name,values in state.items():\n                for key,value in values.items():getattr(self.adapters[name],key).copy_(value.to(self.device))\n\n    def preprocess(self,rgb):\n        width=CFG[\'input_width\']\n        h,w=rgb.shape[-2:]\n        height=round(h*width/w/14)*14\n        image=F.interpolate(rgb.to(self.device),size=(height,width),mode=\'bicubic\',align_corners=False,antialias=True)\n        # Our selected frames are landscape. Avoid silently evaluating cropped GT.\n        assert height<=width, \'Portrait frames need an explicitly aligned crop protocol.\'\n        return image.clamp(0,1)\n\n    def predict(self,image,size):\n        images=image.unsqueeze(0)\n        with torch.autocast(\'cuda\',dtype=self.dtype):\n            tokens,start=self.model.aggregator(images,use_prompt=False,\n                         feature_idx_ls=self.model.depth_head.intermediate_layer_idx)\n        with torch.autocast(\'cuda\',enabled=False):\n            depth,_=self.model.depth_head(float_tree(tokens),images.float(),start)\n        depth=depth.squeeze(0).squeeze(-1)\n        return F.interpolate(depth.unsqueeze(1),size=size,mode=\'bilinear\',align_corners=False).squeeze(1)\n\n    def align(self,pred,condition,mask):\n        with torch.no_grad():\n            scale,shift=alignment.align_depth_to_condition(pred.detach(),condition,mask,\n                                    n_points=CFG[\'align_points\'],seed=47)\n        if not torch.isfinite(scale).all() or not torch.isfinite(shift).all():\n            raise FloatingPointError(\'Non-finite sparse-depth alignment\')\n        return pred*scale.reshape(-1,1,1)+shift.reshape(-1,1,1),scale.detach(),shift.detach()\n\ndef metrics(pred,gt,valid,mask):\n    pred=pred.detach().float().cpu()\n    ans={}\n    for region,select in [(\'all\',valid),(\'observed\',valid&mask),(\'unobserved\',valid&~mask)]:\n        values=pred[select];truth=gt[select]\n        if not len(values):raise ValueError(\'Empty evaluation mask: \'+region)\n        if not torch.isfinite(values).all():raise FloatingPointError(\'Invalid prediction; not dropping invalid pixels.\')\n        error=values-truth\n        ans[region+\'_absrel_pct\']=100*(error.abs()/truth).mean().item()\n        ans[region+\'_rmse\']=error.square().mean().sqrt().item()\n        ans[region+\'_nonpositive_pct\']=100*(values<=0).float().mean().item()\n    return ans\n\ndef cpu_tree(x):\n    if torch.is_tensor(x):return x.detach().cpu()\n    if isinstance(x,dict):return {k:cpu_tree(v) for k,v in x.items()}\n    if isinstance(x,list):return [cpu_tree(v) for v in x]\n    if isinstance(x,tuple):return tuple(cpu_tree(v) for v in x)\n    return x\n\ndef atomic_torch_save(value,path):\n    path=Path(path);temp=path.with_name(path.name+\'.tmp\')\n    torch.save(value,temp);temp.replace(path)\n\ndef experiment(stage,record,method,lr,steps,seed=47,points=100,noise=0.,bank=None,bank_name=None,donor_group=None):\n    """One frame per target run; donor_group alternates controlled variants\n    using ONE shared adapter. Ground truth never enters target gradients except\n    through the explicitly sampled conditioning points."""\n    check_budget()\n    group=donor_group or [record]\n    settings=dict(config_hash=CONFIG_HASH,capa_commit=CAPA_COMMIT,stage=stage,\n        sample=record[\'id\'],input_hashes=[r[\'sha256\'] for r in group],method=method,lr=lr,\n        steps=steps,seed=seed,points=points,noise=noise,bank_name=bank_name,\n        bank_digest=hashlib.sha256(b\'\'.join(\n            x[k].contiguous().numpy().tobytes() for _,x in sorted(bank.items()) for k in [\'U\',\'V\'])).hexdigest() if bank else None)\n    run_id=hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()[:16]\n    folder=RUNS/(stage+\'_\'+record[\'id\']+\'_\'+method+\'_\'+run_id)\n    folder.mkdir(parents=True,exist_ok=True)\n    done=folder/\'complete.json\'\n    if done.exists():\n        status=json.loads(done.read_text())\n        for required in [\'history.csv\',\'result.pt\']:\n            assert (folder/required).exists(), f\'Incomplete saved run {folder}\'\n        print(\'REUSE\',stage,record[\'id\'],method,flush=True)\n        return folder\n    write_json(folder/\'settings.json\',settings)\n    journal(\'run_started\',run_id=run_id,**settings)\n    history=[];completed=0;attempts=0;optimization_seconds=0.\n    optimizer=None;scaler=None;processed=[]\n    try:\n        RUNNER.install(method,seed,bank)\n        groups=RUNNER.groups(lr)\n        params=[p for g in groups for p in g[\'params\']]\n        nparams=sum(p.numel() for p in params)\n        assert {id(p) for p in params}=={id(p) for p in RUNNER.model.parameters() if p.requires_grad}, \'Trainable backbone parameter leak\'\n        expected={\'lora4\':393216,\'donor\':393216,\'lora1\':98304,\'random\':384,\n                  \'learned\':384,\'learned_energy\':384,\'shared_real\':384,\'hybrid\':98688,\'frozen\':0}\n        assert nparams==expected[method], f\'Unexpected parameter budget: {method} {nparams}\'\n        optimizer=torch.optim.AdamW(groups,weight_decay=CFG[\'weight_decay\']) if groups else None\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=RUNNER.dtype==torch.float16,init_scale=1024.)\n        datasets=[]\n        for rec in group:\n            rgb,gt,valid,cond,mask,cond_meta=make_condition(rec,points,noise)\n            with torch.no_grad():image=RUNNER.preprocess(rgb)\n            datasets.append((image,gt,valid,cond.to(RUNNER.device),mask.to(RUNNER.device),mask,cond_meta))\n        write_json(folder/\'conditioning.json\',[x[-1] for x in datasets])\n        checkpoint_path=folder/\'checkpoint.pt\'\n        if checkpoint_path.exists():\n            cp=torch.load(checkpoint_path,map_location=\'cpu\',weights_only=False)\n            RUNNER.restore_adapter(cp[\'adapter\'])\n            if optimizer:optimizer.load_state_dict(cp[\'optimizer\'])\n            scaler.load_state_dict(cp[\'scaler\'])\n            completed=cp[\'completed\'];attempts=cp[\'attempts\'];history=cp[\'history\']\n            optimization_seconds=cp[\'optimization_seconds\']\n            torch.set_rng_state(cp[\'rng_cpu\']);torch.cuda.set_rng_state(cp[\'rng_cuda\'],RUNNER.device)\n            print(\'RESUME\',record[\'id\'],method,\'at\',completed,flush=True)\n        torch.cuda.reset_peak_memory_stats(RUNNER.device)\n        wall_start=time.perf_counter()\n        eval_steps={0,1,5,10,20,30,50,60,75,100,steps}\n        while completed<=steps:\n            check_budget()\n            final=completed==steps\n            if optimizer:optimizer.zero_grad(set_to_none=True)\n            image,gt,valid,cond,mask_gpu,mask_cpu,cond_meta=datasets[completed%len(datasets)]\n            torch.cuda.synchronize();t0=time.perf_counter()\n            with torch.set_grad_enabled(not final and optimizer is not None):\n                raw=RUNNER.predict(image,tuple(cond.shape[-2:]))\n                if not torch.isfinite(raw).all():raise FloatingPointError(\'Non-finite raw depth\')\n                pred,scale,shift=RUNNER.align(raw,cond,mask_gpu)\n                loss=(pred-cond).abs()[mask_gpu].mean()\n                if not torch.isfinite(loss):raise FloatingPointError(\'Non-finite sparse loss\')\n            torch.cuda.synchronize();forward_seconds=time.perf_counter()-t0\n            # CPU metrics and disk writes are deliberately excluded from the\n            # recorded optimization timer. These are instrumented pilot timings.\n            if completed in eval_steps:\n                row=dict(updates=completed,attempts=attempts,sparse_L1=float(loss.detach()),\n                         adaptation_seconds=optimization_seconds,\n                         time_to_prediction_seconds=optimization_seconds+forward_seconds,\n                         trainable_parameters=nparams,scale=float(scale[0]),shift=float(shift[0]),\n                         **metrics(pred,gt,valid,mask_cpu))\n                history=[h for h in history if h[\'updates\']!=completed]+[row]\n                pd.DataFrame(history).sort_values(\'updates\').to_csv(folder/\'history.csv\',index=False)\n                print(f\'{stage} {record["id"]} {method}: {completed}/{steps}, \'\n                      f\'L1={row["sparse_L1"]:.4f}, unobserved={row["unobserved_absrel_pct"]:.3f}%\',flush=True)\n            if final:\n                result=dict(adapter=RUNNER.adapter_state(),prediction=pred.detach().float().cpu(),\n                    history=history,settings=settings,trainable_parameters=nparams,\n                    peak_memory_GB=torch.cuda.max_memory_allocated(RUNNER.device)/1024**3,\n                    optimization_seconds=optimization_seconds,final_forward_seconds=forward_seconds,\n                    elapsed_segment_seconds=time.perf_counter()-wall_start)\n                atomic_torch_save(result,folder/\'result.pt\')\n                write_json(done,dict(status=\'complete\',run_id=run_id,finished=utcnow(),\n                                    trainable_parameters=nparams,peak_memory_GB=result[\'peak_memory_GB\']))\n                journal(\'run_complete\',run_id=run_id,folder=str(folder.relative_to(ROOT)))\n                if sum(1 for _ in RUNS.glob(\'*/complete.json\')) % 5 == 0:\n                    backup()\n                return folder\n            torch.cuda.synchronize();back_start=time.perf_counter()\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            norm=torch.nn.utils.clip_grad_norm_(params,1.)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            torch.cuda.synchronize()\n            optimization_seconds+=forward_seconds+time.perf_counter()-back_start\n            attempts+=1\n            success=scaler.get_scale()>=old_scale\n            if success:completed+=1\n            del raw,pred,loss\n            optimizer.zero_grad(set_to_none=True)\n            if success and (completed%10==0 or completed==steps):\n                atomic_torch_save(dict(adapter=RUNNER.adapter_state(),optimizer=cpu_tree(optimizer.state_dict()),\n                    scaler=scaler.state_dict(),completed=completed,attempts=attempts,history=history,\n                    optimization_seconds=optimization_seconds,rng_cpu=torch.get_rng_state(),\n                    rng_cuda=torch.cuda.get_rng_state(RUNNER.device)),checkpoint_path)\n            if attempts>steps+40:raise FloatingPointError(\'Too many skipped FP16 updates\')\n    except Exception as exc:\n        write_json(folder/\'failure.json\',dict(error=str(exc),traceback=traceback.format_exc(),at=utcnow()))\n        journal(\'run_failed\',run_id=run_id,error=str(exc))\n        raise\n    finally:\n        optimizer=None;scaler=None;processed=[]\n        # Local graph references must disappear before the next run.\n        gc.collect();torch.cuda.empty_cache()\n\nRUNNER=Runner()\n# Forward-equivalence smoke test: every zero-initialized adapter must leave\n# pretrained output unchanged, and an update must not touch any base weights.\nsmoke_record=records_for(\'validation\')[0]\nrgb_smoke,_,_=load_frame(smoke_record)\nwith torch.no_grad():\n    image_smoke=RUNNER.preprocess(rgb_smoke)\n    RUNNER.install(\'frozen\',47)\n    reference=RUNNER.predict(image_smoke,tuple(rgb_smoke.shape[-2:])).cpu()\n    RUNNER.install(\'lora4\',47)\n    check=RUNNER.predict(image_smoke,tuple(rgb_smoke.shape[-2:])).cpu()\nassert torch.allclose(reference,check,atol=1e-4,rtol=1e-3), \'Zero-adapter equivalence failed\'\nRUNNER.clear()\ndel reference,check,image_smoke,rgb_smoke\nprint(\'Zero-initialized LoRA forward-equivalence check passed.\')\nbackup()\n', encoding="utf-8")
import random, math, traceback, importlib.util
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# Import VGGT directly. This avoids CAPA's package/PEFT/TorchAO import chain.
sys.path.insert(0,str(REPO/'third_party'/'VGGT_VPT'))
from vggt.models.vggt import VGGT
spec = importlib.util.spec_from_file_location('da4022_alignment',REPO/'capa'/'utils'/'alignment.py')
alignment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(alignment)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def float_tree(x):
    if torch.is_tensor(x): return x.float()
    if isinstance(x,list): return [float_tree(v) for v in x]
    if isinstance(x,tuple): return tuple(float_tree(v) for v in x)
    return x

def checkpoint_forward(original):
    def forward(*args,**kwargs):
        if torch.is_grad_enabled():
            return checkpoint(original,*args,use_reentrant=False,**kwargs)
        return original(*args,**kwargs)
    return forward

class AdaptedLinear(nn.Module):
    def __init__(self,base,method,rank=4,bases=None,residual_rank=1):
        super().__init__()
        self.base=base; self.method=method
        dev=base.weight.device
        out_features,in_features=base.weight.shape
        if method in ['lora4','lora1','donor']:
            self.A=nn.Parameter(torch.empty(rank,in_features,device=dev,dtype=torch.float32))
            self.B=nn.Parameter(torch.zeros(out_features,rank,device=dev,dtype=torch.float32))
            nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        else:
            U,V=bases
            self.register_buffer('U',U.to(dev).float())
            self.register_buffer('V',V.to(dev).float())
            self.M=nn.Parameter(torch.zeros(U.shape[1],V.shape[1],device=dev,dtype=torch.float32))
            if method=='hybrid':
                self.A=nn.Parameter(torch.empty(residual_rank,in_features,device=dev,dtype=torch.float32))
                self.B=nn.Parameter(torch.zeros(out_features,residual_rank,device=dev,dtype=torch.float32))
                nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        self.base.requires_grad_(False)

    def forward(self,x):
        y=self.base(x)
        if hasattr(self,'M'):
            y=y+2*F.linear(F.linear(F.linear(x,self.V.T),self.M),self.U)
        if hasattr(self,'A'):
            y=y+2*F.linear(F.linear(x,self.A),self.B)
        return y

class Runner:
    def __init__(self):
        self.device=torch.device('cuda:0')
        self.dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        print('Loading VGGT checkpoint. First download can take several minutes.',flush=True)
        self.model=VGGT.from_pretrained('facebook/VGGT-1B').eval().requires_grad_(False).to(self.device)
        self.names=[name for name,layer in self.model.named_modules()
                    if isinstance(layer,nn.Linear) and 'aggregator.patch_embed.' in name and name.endswith('.qkv')]
        assert len(self.names)==24, f'Unexpected VGGT layout: {len(self.names)} image-encoder QKV layers.'
        for block in self.model.modules():
            if block.__class__.__name__ in {'Block','NestedTensorBlock'}:
                block.forward=checkpoint_forward(block.forward)
        self.adapters={}
        print('Loaded. Precision:',self.dtype,'target layers:',len(self.names),flush=True)

    def clear(self):
        for name,layer in list(self.adapters.items()):
            parent,child=name.rsplit('.',1)
            setattr(self.model.get_submodule(parent),child,layer.base)
        self.adapters={}
        self.model.requires_grad_(False)
        gc.collect();torch.cuda.empty_cache()

    def install(self,method,seed,banks=None):
        self.clear();seed_everything(seed)
        if method=='frozen':return
        for i,name in enumerate(self.names):
            base=self.model.get_submodule(name)
            basis=None
            rank=1 if method=='lora1' else CFG['rank']
            if method=='random':
                g=torch.Generator().manual_seed(seed+76+i*1009)
                U=torch.linalg.qr(torch.randn(base.out_features,CFG['basis_dim'],generator=g),mode='reduced').Q
                V=torch.linalg.qr(torch.randn(base.in_features,CFG['basis_dim'],generator=g),mode='reduced').Q
                basis=(U,V)
            elif method not in ['donor','lora4','lora1']:
                assert banks is not None and name in banks, f'Missing bases for {name}'
                basis=(banks[name]['U'],banks[name]['V'])
            layer=AdaptedLinear(base,method,rank,basis,CFG['residual_rank'])
            parent,child=name.rsplit('.',1)
            setattr(self.model.get_submodule(parent),child,layer)
            self.adapters[name]=layer
        self.model.eval()

    def groups(self,lr):
        groups=[]
        for layer in self.adapters.values():
            if hasattr(layer,'M'):groups.append({'params':[layer.M],'lr':lr})
            if hasattr(layer,'A'):
                rate=CFG['hybrid_residual_lr'] if layer.method=='hybrid' else lr
                groups.append({'params':[layer.A,layer.B],'lr':rate})
        return groups

    def adapter_state(self):
        return {name:{key:p.detach().cpu().clone() for key,p in layer.named_parameters(recurse=False)}
                for name,layer in self.adapters.items()}

    def restore_adapter(self,state):
        assert set(state)==set(self.adapters)
        with torch.no_grad():
            for name,values in state.items():
                for key,value in values.items():getattr(self.adapters[name],key).copy_(value.to(self.device))

    def preprocess(self,rgb):
        width=CFG['input_width']
        h,w=rgb.shape[-2:]
        height=round(h*width/w/14)*14
        image=F.interpolate(rgb.to(self.device),size=(height,width),mode='bicubic',align_corners=False,antialias=True)
        # Our selected frames are landscape. Avoid silently evaluating cropped GT.
        assert height<=width, 'Portrait frames need an explicitly aligned crop protocol.'
        return image.clamp(0,1)

    def predict(self,image,size):
        images=image.unsqueeze(0)
        with torch.autocast('cuda',dtype=self.dtype):
            tokens,start=self.model.aggregator(images,use_prompt=False,
                         feature_idx_ls=self.model.depth_head.intermediate_layer_idx)
        with torch.autocast('cuda',enabled=False):
            depth,_=self.model.depth_head(float_tree(tokens),images.float(),start)
        depth=depth.squeeze(0).squeeze(-1)
        return F.interpolate(depth.unsqueeze(1),size=size,mode='bilinear',align_corners=False).squeeze(1)

    def align(self,pred,condition,mask):
        with torch.no_grad():
            scale,shift=alignment.align_depth_to_condition(pred.detach(),condition,mask,
                                    n_points=CFG['align_points'],seed=47)
        if not torch.isfinite(scale).all() or not torch.isfinite(shift).all():
            raise FloatingPointError('Non-finite sparse-depth alignment')
        return pred*scale.reshape(-1,1,1)+shift.reshape(-1,1,1),scale.detach(),shift.detach()

def metrics(pred,gt,valid,mask):
    pred=pred.detach().float().cpu()
    ans={}
    for region,select in [('all',valid),('observed',valid&mask),('unobserved',valid&~mask)]:
        values=pred[select];truth=gt[select]
        if not len(values):raise ValueError('Empty evaluation mask: '+region)
        if not torch.isfinite(values).all():raise FloatingPointError('Invalid prediction; not dropping invalid pixels.')
        error=values-truth
        ans[region+'_absrel_pct']=100*(error.abs()/truth).mean().item()
        ans[region+'_rmse']=error.square().mean().sqrt().item()
        ans[region+'_nonpositive_pct']=100*(values<=0).float().mean().item()
    return ans

def cpu_tree(x):
    if torch.is_tensor(x):return x.detach().cpu()
    if isinstance(x,dict):return {k:cpu_tree(v) for k,v in x.items()}
    if isinstance(x,list):return [cpu_tree(v) for v in x]
    if isinstance(x,tuple):return tuple(cpu_tree(v) for v in x)
    return x

def atomic_torch_save(value,path):
    path=Path(path);temp=path.with_name(path.name+'.tmp')
    torch.save(value,temp);temp.replace(path)

def experiment(stage,record,method,lr,steps,seed=47,points=100,noise=0.,bank=None,bank_name=None,donor_group=None):
    """One frame per target run; donor_group alternates controlled variants
    using ONE shared adapter. Ground truth never enters target gradients except
    through the explicitly sampled conditioning points."""
    check_budget()
    group=donor_group or [record]
    settings=dict(config_hash=CONFIG_HASH,capa_commit=CAPA_COMMIT,stage=stage,
        sample=record['id'],input_hashes=[r['sha256'] for r in group],method=method,lr=lr,
        steps=steps,seed=seed,points=points,noise=noise,bank_name=bank_name,
        bank_digest=hashlib.sha256(b''.join(
            x[k].contiguous().numpy().tobytes() for _,x in sorted(bank.items()) for k in ['U','V'])).hexdigest() if bank else None)
    run_id=hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()[:16]
    folder=RUNS/(stage+'_'+record['id']+'_'+method+'_'+run_id)
    folder.mkdir(parents=True,exist_ok=True)
    done=folder/'complete.json'
    if done.exists():
        status=json.loads(done.read_text())
        for required in ['history.csv','result.pt']:
            assert (folder/required).exists(), f'Incomplete saved run {folder}'
        print('REUSE',stage,record['id'],method,flush=True)
        return folder
    write_json(folder/'settings.json',settings)
    journal('run_started',run_id=run_id,**settings)
    history=[];completed=0;attempts=0;optimization_seconds=0.
    optimizer=None;scaler=None;processed=[]
    try:
        RUNNER.install(method,seed,bank)
        groups=RUNNER.groups(lr)
        params=[p for g in groups for p in g['params']]
        nparams=sum(p.numel() for p in params)
        assert {id(p) for p in params}=={id(p) for p in RUNNER.model.parameters() if p.requires_grad}, 'Trainable backbone parameter leak'
        expected={'lora4':393216,'donor':393216,'lora1':98304,'random':384,
                  'learned':384,'learned_energy':384,'shared_real':384,'hybrid':98688,'frozen':0}
        assert nparams==expected[method], f'Unexpected parameter budget: {method} {nparams}'
        optimizer=torch.optim.AdamW(groups,weight_decay=CFG['weight_decay']) if groups else None
        scaler=torch.amp.GradScaler('cuda',enabled=RUNNER.dtype==torch.float16,init_scale=1024.)
        datasets=[]
        for rec in group:
            rgb,gt,valid,cond,mask,cond_meta=make_condition(rec,points,noise)
            with torch.no_grad():image=RUNNER.preprocess(rgb)
            datasets.append((image,gt,valid,cond.to(RUNNER.device),mask.to(RUNNER.device),mask,cond_meta))
        write_json(folder/'conditioning.json',[x[-1] for x in datasets])
        checkpoint_path=folder/'checkpoint.pt'
        if checkpoint_path.exists():
            cp=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
            RUNNER.restore_adapter(cp['adapter'])
            if optimizer:optimizer.load_state_dict(cp['optimizer'])
            scaler.load_state_dict(cp['scaler'])
            completed=cp['completed'];attempts=cp['attempts'];history=cp['history']
            optimization_seconds=cp['optimization_seconds']
            torch.set_rng_state(cp['rng_cpu']);torch.cuda.set_rng_state(cp['rng_cuda'],RUNNER.device)
            print('RESUME',record['id'],method,'at',completed,flush=True)
        torch.cuda.reset_peak_memory_stats(RUNNER.device)
        wall_start=time.perf_counter()
        eval_steps={0,1,5,10,20,30,50,60,75,100,steps}
        while completed<=steps:
            check_budget()
            final=completed==steps
            if optimizer:optimizer.zero_grad(set_to_none=True)
            image,gt,valid,cond,mask_gpu,mask_cpu,cond_meta=datasets[completed%len(datasets)]
            torch.cuda.synchronize();t0=time.perf_counter()
            with torch.set_grad_enabled(not final and optimizer is not None):
                raw=RUNNER.predict(image,tuple(cond.shape[-2:]))
                if not torch.isfinite(raw).all():raise FloatingPointError('Non-finite raw depth')
                pred,scale,shift=RUNNER.align(raw,cond,mask_gpu)
                loss=(pred-cond).abs()[mask_gpu].mean()
                if not torch.isfinite(loss):raise FloatingPointError('Non-finite sparse loss')
            torch.cuda.synchronize();forward_seconds=time.perf_counter()-t0
            # CPU metrics and disk writes are deliberately excluded from the
            # recorded optimization timer. These are instrumented pilot timings.
            if completed in eval_steps:
                row=dict(updates=completed,attempts=attempts,sparse_L1=float(loss.detach()),
                         adaptation_seconds=optimization_seconds,
                         time_to_prediction_seconds=optimization_seconds+forward_seconds,
                         trainable_parameters=nparams,scale=float(scale[0]),shift=float(shift[0]),
                         **metrics(pred,gt,valid,mask_cpu))
                history=[h for h in history if h['updates']!=completed]+[row]
                pd.DataFrame(history).sort_values('updates').to_csv(folder/'history.csv',index=False)
                print(f'{stage} {record["id"]} {method}: {completed}/{steps}, '
                      f'L1={row["sparse_L1"]:.4f}, unobserved={row["unobserved_absrel_pct"]:.3f}%',flush=True)
            if final:
                result=dict(adapter=RUNNER.adapter_state(),prediction=pred.detach().float().cpu(),
                    history=history,settings=settings,trainable_parameters=nparams,
                    peak_memory_GB=torch.cuda.max_memory_allocated(RUNNER.device)/1024**3,
                    optimization_seconds=optimization_seconds,final_forward_seconds=forward_seconds,
                    elapsed_segment_seconds=time.perf_counter()-wall_start)
                atomic_torch_save(result,folder/'result.pt')
                write_json(done,dict(status='complete',run_id=run_id,finished=utcnow(),
                                    trainable_parameters=nparams,peak_memory_GB=result['peak_memory_GB']))
                journal('run_complete',run_id=run_id,folder=str(folder.relative_to(ROOT)))
                if sum(1 for _ in RUNS.glob('*/complete.json')) % 5 == 0:
                    backup()
                return folder
            torch.cuda.synchronize();back_start=time.perf_counter()
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            norm=torch.nn.utils.clip_grad_norm_(params,1.)
            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()
            torch.cuda.synchronize()
            optimization_seconds+=forward_seconds+time.perf_counter()-back_start
            attempts+=1
            success=scaler.get_scale()>=old_scale
            if success:completed+=1
            del raw,pred,loss
            optimizer.zero_grad(set_to_none=True)
            if success and (completed%10==0 or completed==steps):
                atomic_torch_save(dict(adapter=RUNNER.adapter_state(),optimizer=cpu_tree(optimizer.state_dict()),
                    scaler=scaler.state_dict(),completed=completed,attempts=attempts,history=history,
                    optimization_seconds=optimization_seconds,rng_cpu=torch.get_rng_state(),
                    rng_cuda=torch.cuda.get_rng_state(RUNNER.device)),checkpoint_path)
            if attempts>steps+40:raise FloatingPointError('Too many skipped FP16 updates')
    except Exception as exc:
        write_json(folder/'failure.json',dict(error=str(exc),traceback=traceback.format_exc(),at=utcnow()))
        journal('run_failed',run_id=run_id,error=str(exc))
        raise
    finally:
        optimizer=None;scaler=None;processed=[]
        # Local graph references must disappear before the next run.
        gc.collect();torch.cuda.empty_cache()

RUNNER=Runner()
# Forward-equivalence smoke test: every zero-initialized adapter must leave
# pretrained output unchanged, and an update must not touch any base weights.
smoke_record=records_for('validation')[0]
rgb_smoke,_,_=load_frame(smoke_record)
with torch.no_grad():
    image_smoke=RUNNER.preprocess(rgb_smoke)
    RUNNER.install('frozen',47)
    reference=RUNNER.predict(image_smoke,tuple(rgb_smoke.shape[-2:])).cpu()
    RUNNER.install('lora4',47)
    check=RUNNER.predict(image_smoke,tuple(rgb_smoke.shape[-2:])).cpu()
assert torch.allclose(reference,check,atol=1e-4,rtol=1e-3), 'Zero-adapter equivalence failed'
RUNNER.clear()
del reference,check,image_smoke,rgb_smoke
print('Zero-initialized LoRA forward-equivalence check passed.')
backup()


## 7. GPU preflight: two updates for every method and checkpoint restoration

In [ ]:
(ROOT / "code" / '06b_gpu_preflight.py').write_text("# This gate runs BEFORE donor training and the long experiments. The random\n# placeholder bases below are ONLY for testing execution, not learned priors.\npreflight_record=records_for('validation')[0]\nRUNNER.clear()\nplaceholder_bank={}\nfor i,name in enumerate(RUNNER.names):\n    base=RUNNER.model.get_submodule(name)\n    generator=torch.Generator().manual_seed(700+i)\n    placeholder_bank[name]={\n        'U':torch.linalg.qr(torch.randn(base.out_features,CFG['basis_dim'],generator=generator),mode='reduced').Q,\n        'V':torch.linalg.qr(torch.randn(base.in_features,CFG['basis_dim'],generator=generator),mode='reduced').Q,\n    }\n\npreflight_folders={}\nfor method in ['lora4','lora1','learned','random','hybrid','shared_real','learned_energy']:\n    uses_bank=method not in ['lora4','lora1','random']\n    folder=experiment('gpu_preflight',preflight_record,method,\n        .01 if uses_bank or method=='random' else .001,\n        steps=2,bank=placeholder_bank if uses_bank else None,\n        bank_name='preflight_random_placeholder' if uses_bank else None)\n    result=torch.load(folder/'result.pt',map_location='cpu',weights_only=False)\n    assert result['history'][-1]['updates']==2, 'Two successful updates not completed'\n    assert all(torch.isfinite(t).all() for layer in result['adapter'].values() for t in layer.values()), 'Invalid adapter weights'\n    # Every adapted layer should receive at least some gradient signal.\n    active=sum(float(layer.get('B',torch.zeros(1)).abs().sum())+\n               float(layer.get('M',torch.zeros(1)).abs().sum()) for layer in result['adapter'].values())\n    assert active>0, 'Optimizer did not change any zero-initialized correction'\n    assert np.isfinite(result['history'][-1]['unobserved_absrel_pct'])\n    preflight_folders[method]=folder\n\n# Verify that the saved adapter/optimizer/scaler checkpoint can actually be\n# restored into fresh objects, and that it reproduces the final prediction.\nfolder=preflight_folders['lora4']\nsaved=torch.load(folder/'checkpoint.pt',map_location='cpu',weights_only=False)\nfinished=torch.load(folder/'result.pt',map_location='cpu',weights_only=False)\nRUNNER.install('lora4',47)\nRUNNER.restore_adapter(saved['adapter'])\ntest_optimizer=torch.optim.AdamW(RUNNER.groups(.001),weight_decay=0.)\ntest_optimizer.load_state_dict(saved['optimizer'])\ntest_scaler=torch.amp.GradScaler('cuda',enabled=RUNNER.dtype==torch.float16)\ntest_scaler.load_state_dict(saved['scaler'])\nimage,truth,valid,condition,mask,_=make_condition(preflight_record,100,0.)\nwith torch.no_grad():\n    processed=RUNNER.preprocess(image)\n    raw=RUNNER.predict(processed,tuple(truth.shape[-2:]))\n    restored,_,_=RUNNER.align(raw,condition.to(RUNNER.device),mask.to(RUNNER.device))\nassert torch.allclose(restored.cpu(),finished['prediction'],atol=2e-3,rtol=2e-3), 'Checkpoint prediction round-trip failed'\ntest_optimizer=None;test_scaler=None\ndel processed,raw,restored,result,saved,finished,placeholder_bank\nRUNNER.clear();gc.collect();torch.cuda.empty_cache()\nwrite_json(ROOT/'GPU_PREFLIGHT_PASSED.json',dict(time=utcnow(),methods=list(preflight_folders),\n    updates_per_method=2,checkpoint_roundtrip=True,capa_commit=CAPA_COMMIT,config_hash=CONFIG_HASH))\njournal('gpu_preflight_passed',methods=list(preflight_folders))\nprint('\\nGPU PREFLIGHT PASSED: all seven methods completed forward/backward/update, finite metrics, saving and checkpoint restoration.')\nprint('The full experiment cells may now run. This is execution validation, not evidence of model quality.')\nbackup()\n", encoding="utf-8")
# This gate runs BEFORE donor training and the long experiments. The random
# placeholder bases below are ONLY for testing execution, not learned priors.
preflight_record=records_for('validation')[0]
RUNNER.clear()
placeholder_bank={}
for i,name in enumerate(RUNNER.names):
    base=RUNNER.model.get_submodule(name)
    generator=torch.Generator().manual_seed(700+i)
    placeholder_bank[name]={
        'U':torch.linalg.qr(torch.randn(base.out_features,CFG['basis_dim'],generator=generator),mode='reduced').Q,
        'V':torch.linalg.qr(torch.randn(base.in_features,CFG['basis_dim'],generator=generator),mode='reduced').Q,
    }

preflight_folders={}
for method in ['lora4','lora1','learned','random','hybrid','shared_real','learned_energy']:
    uses_bank=method not in ['lora4','lora1','random']
    folder=experiment('gpu_preflight',preflight_record,method,
        .01 if uses_bank or method=='random' else .001,
        steps=2,bank=placeholder_bank if uses_bank else None,
        bank_name='preflight_random_placeholder' if uses_bank else None)
    result=torch.load(folder/'result.pt',map_location='cpu',weights_only=False)
    assert result['history'][-1]['updates']==2, 'Two successful updates not completed'
    assert all(torch.isfinite(t).all() for layer in result['adapter'].values() for t in layer.values()), 'Invalid adapter weights'
    # Every adapted layer should receive at least some gradient signal.
    active=sum(float(layer.get('B',torch.zeros(1)).abs().sum())+
               float(layer.get('M',torch.zeros(1)).abs().sum()) for layer in result['adapter'].values())
    assert active>0, 'Optimizer did not change any zero-initialized correction'
    assert np.isfinite(result['history'][-1]['unobserved_absrel_pct'])
    preflight_folders[method]=folder

# Verify that the saved adapter/optimizer/scaler checkpoint can actually be
# restored into fresh objects, and that it reproduces the final prediction.
folder=preflight_folders['lora4']
saved=torch.load(folder/'checkpoint.pt',map_location='cpu',weights_only=False)
finished=torch.load(folder/'result.pt',map_location='cpu',weights_only=False)
RUNNER.install('lora4',47)
RUNNER.restore_adapter(saved['adapter'])
test_optimizer=torch.optim.AdamW(RUNNER.groups(.001),weight_decay=0.)
test_optimizer.load_state_dict(saved['optimizer'])
test_scaler=torch.amp.GradScaler('cuda',enabled=RUNNER.dtype==torch.float16)
test_scaler.load_state_dict(saved['scaler'])
image,truth,valid,condition,mask,_=make_condition(preflight_record,100,0.)
with torch.no_grad():
    processed=RUNNER.preprocess(image)
    raw=RUNNER.predict(processed,tuple(truth.shape[-2:]))
    restored,_,_=RUNNER.align(raw,condition.to(RUNNER.device),mask.to(RUNNER.device))
assert torch.allclose(restored.cpu(),finished['prediction'],atol=2e-3,rtol=2e-3), 'Checkpoint prediction round-trip failed'
test_optimizer=None;test_scaler=None
del processed,raw,restored,result,saved,finished,placeholder_bank
RUNNER.clear();gc.collect();torch.cuda.empty_cache()
write_json(ROOT/'GPU_PREFLIGHT_PASSED.json',dict(time=utcnow(),methods=list(preflight_folders),
    updates_per_method=2,checkpoint_roundtrip=True,capa_commit=CAPA_COMMIT,config_hash=CONFIG_HASH))
journal('gpu_preflight_passed',methods=list(preflight_folders))
print('\nGPU PREFLIGHT PASSED: all seven methods completed forward/backward/update, finite metrics, saving and checkpoint restoration.')
print('The full experiment cells may now run. This is execution validation, not evidence of model quality.')
backup()


## 8. Train donor adapters and extract attribute/shared bases

In [ ]:
(ROOT / "code" / '07_donors_bases.py').write_text('SESSION_START=time.monotonic()  # GPU-stage wall-time budget starts after data/model loading.\nSTAGE_STATUS={}\nBANKS={}\n\ndef load_result(folder):\n    return torch.load(Path(folder)/\'result.pt\',map_location=\'cpu\',weights_only=False)\n\ndef extract_energy_basis(folders,label):\n    """Top left/right uncentered update-energy subspaces. Each update is\n    Frobenius-normalized. This is a simple pooling baseline, not a replication\n    of Jiang et al.\'s robust shared-mean solver."""\n    states=[load_result(p)[\'adapter\'] for p in folders]\n    assert states\n    bank={};diagnostics=[]\n    for name in RUNNER.names:\n        left,right=[],[]\n        for state in states:\n            A,B=state[name][\'A\'].float(),state[name][\'B\'].float()\n            qB,rB=torch.linalg.qr(B,mode=\'reduced\')\n            qA,rA=torch.linalg.qr(A.T,mode=\'reduced\')\n            u,s,vh=torch.linalg.svd(rB@rA.T,full_matrices=False)\n            assert torch.isfinite(s).all() and s.norm()>1e-10, \'Empty/invalid donor update\'\n            # Normalize each effective donor update, avoiding one large adapter\n            # dominating solely because of its displacement magnitude.\n            normalized=s/s.norm()\n            left.append((qB@u)*normalized)\n            right.append((qA@vh.T)*normalized)\n        u,sl,_=torch.linalg.svd(torch.cat(left,dim=1),full_matrices=False)\n        v,sr,_=torch.linalg.svd(torch.cat(right,dim=1),full_matrices=False)\n        d=CFG[\'basis_dim\']\n        assert min(len(sl),len(sr))>=d\n        U,V=u[:,:d].contiguous(),v[:,:d].contiguous()\n        assert torch.allclose(U.T@U,torch.eye(d),atol=2e-5)\n        assert torch.allclose(V.T@V,torch.eye(d),atol=2e-5)\n        bank[name]={\'U\':U,\'V\':V}\n        diagnostics.append(dict(bank=label,layer=name,left_retained_energy=float(sl[:d].square().sum()/sl.square().sum()),\n            right_retained_energy=float(sr[:d].square().sum()/sr.square().sum())))\n    folder=ROOT/\'bases\';folder.mkdir(exist_ok=True)\n    atomic_torch_save(bank,folder/(label+\'.pt\'))\n    pd.DataFrame(diagnostics).to_csv(folder/(label+\'_energy.csv\'),index=False)\n    write_json(folder/(label+\'_sources.json\'),dict(label=label,source_runs=[str(p.relative_to(ROOT)) for p in folders],\n        extraction=\'Frobenius-normalized update energy, independently pooled left and right singular subspaces\',basis_dim=CFG[\'basis_dim\']))\n    return bank\n\ndef extract_robust_basis(folders,label):\n    """IRLS rank-d shared mean of effective updates, using low-rank factors.\n    The robust p=1 objective is approximated for a fixed number of iterations.\n    Store orthonormal singular bases; only target coefficients are optimized."""\n    states=[load_result(p)[\'adapter\'] for p in folders]\n    bank={};diagnostics=[]\n    for name in RUNNER.names:\n        pairs=[(s[name][\'A\'].float(),2*s[name][\'B\'].float()) for s in states]\n        norms=torch.tensor([float(((B.T@B)*(A@A.T).T).sum()) for A,B in pairs]).clamp_min(0)\n        eps=max(float(norms.sqrt().median())*1e-3,1e-8)\n        weights=torch.ones(len(pairs))/len(pairs)\n        for iteration in range(CFG[\'irls_iterations\']):\n            weights=weights/weights.sum()\n            L=torch.cat([B*weights[i].sqrt() for i,(A,B) in enumerate(pairs)],dim=1)\n            R=torch.cat([A.T*weights[i].sqrt() for i,(A,B) in enumerate(pairs)],dim=1)\n            qL,rL=torch.linalg.qr(L,mode=\'reduced\')\n            qR,rR=torch.linalg.qr(R,mode=\'reduced\')\n            u,sv,vh=torch.linalg.svd(rL@rR.T,full_matrices=False)\n            d=CFG[\'basis_dim\'];s=sv[:d];U=qL@u[:,:d];V=qR@vh[:d].T\n            residuals=[]\n            for i,(A,B) in enumerate(pairs):\n                cross=((U.T@B)@(A@V)).diagonal().mul(s).sum()\n                residuals.append((norms[i]+s.square().sum()-2*cross).clamp_min(0))\n            residuals=torch.stack(residuals)\n            if iteration+1<CFG[\'irls_iterations\']:\n                weights=1/torch.sqrt(residuals+eps**2)\n        assert torch.isfinite(U).all() and torch.isfinite(V).all()\n        assert torch.allclose(U.T@U,torch.eye(d),atol=2e-5)\n        assert torch.allclose(V.T@V,torch.eye(d),atol=2e-5)\n        bank[name]={\'U\':U.contiguous(),\'V\':V.contiguous()}\n        for i in range(len(pairs)):\n            diagnostics.append(dict(bank=label,layer=name,donor_index=i,weight=float(weights[i]/weights.sum()),\n                                    residual_norm=float(residuals[i].sqrt())))\n    folder=ROOT/\'bases\';folder.mkdir(exist_ok=True)\n    atomic_torch_save(bank,folder/(label+\'.pt\'))\n    pd.DataFrame(diagnostics).to_csv(folder/(label+\'_irls.csv\'),index=False)\n    write_json(folder/(label+\'_sources.json\'),dict(label=label,source_runs=[str(p.relative_to(ROOT)) for p in folders],\n        extraction=\'Rank-d weighted shared update mean with p=1 IRLS; orthonormal left/right singular bases\',\n        iterations=CFG[\'irls_iterations\'],basis_dim=CFG[\'basis_dim\']))\n    return bank\n\ndef train_donors_and_bases():\n    synthetic={attribute:[] for attribute in CFG[\'synthetic_attributes\']}\n    for task in SYNTH_TASKS:\n        check_budget()\n        folder=experiment(\'synthetic_donor\',task[\'frames\'][0],\'donor\',CFG[\'donor_lr\'],\n            CFG[\'donor_steps\'],points=500,donor_group=task[\'frames\'])\n        synthetic[task[\'attribute\']].append(folder)\n    for attribute,folders in synthetic.items():\n        BANKS[attribute]=extract_robust_basis(folders,attribute)\n    all_synthetic=[p for folders in synthetic.values() for p in folders]\n    BANKS[\'combined\']=extract_robust_basis(all_synthetic,\'combined\')\n    BANKS[\'combined_energy\']=extract_energy_basis(all_synthetic,\'combined_energy\')\n    real_folders=[]\n    for scene in CFG[\'split\'][\'donor\']:\n        group=[r for r in records_for(\'donor\') if r[\'scene\']==scene]\n        real_folders.append(experiment(\'real_donor\',group[0],\'donor\',CFG[\'donor_lr\'],\n            CFG[\'donor_steps\'],points=100,donor_group=group))\n    BANKS[\'shared_real\']=extract_robust_basis(real_folders,\'shared_real\')\n    # Fixed/hybrid zero-output checks, without using target GT.\n    rec=records_for(\'validation\')[0]\n    rgb,_,_=load_frame(rec)\n    with torch.no_grad():\n        image=RUNNER.preprocess(rgb)\n        RUNNER.install(\'frozen\',47);reference=RUNNER.predict(image,tuple(rgb.shape[-2:])).cpu()\n        for method in [\'learned\',\'hybrid\',\'random\']:\n            RUNNER.install(method,47,BANKS[\'combined\'] if method!=\'random\' else None)\n            output=RUNNER.predict(image,tuple(rgb.shape[-2:])).cpu()\n            assert torch.allclose(reference,output,atol=1e-4,rtol=1e-3), \'Zero-output invariant failed: \'+method\n    RUNNER.clear()\n    STAGE_STATUS[\'donors\']=\'complete\'\n    journal(\'bases_ready\',banks=list(BANKS))\n\ntry:\n    train_donors_and_bases()\nexcept BudgetStop as exc:\n    STAGE_STATUS[\'donors\']=\'budget_stopped\';print(exc)\nexcept Exception:\n    STAGE_STATUS[\'donors\']=\'failed\';backup();raise\nfinally:\n    write_json(ROOT/\'stage_status.json\',STAGE_STATUS)\nbackup()\n', encoding="utf-8")
SESSION_START=time.monotonic()  # GPU-stage wall-time budget starts after data/model loading.
STAGE_STATUS={}
BANKS={}

def load_result(folder):
    return torch.load(Path(folder)/'result.pt',map_location='cpu',weights_only=False)

def extract_energy_basis(folders,label):
    """Top left/right uncentered update-energy subspaces. Each update is
    Frobenius-normalized. This is a simple pooling baseline, not a replication
    of Jiang et al.'s robust shared-mean solver."""
    states=[load_result(p)['adapter'] for p in folders]
    assert states
    bank={};diagnostics=[]
    for name in RUNNER.names:
        left,right=[],[]
        for state in states:
            A,B=state[name]['A'].float(),state[name]['B'].float()
            qB,rB=torch.linalg.qr(B,mode='reduced')
            qA,rA=torch.linalg.qr(A.T,mode='reduced')
            u,s,vh=torch.linalg.svd(rB@rA.T,full_matrices=False)
            assert torch.isfinite(s).all() and s.norm()>1e-10, 'Empty/invalid donor update'
            # Normalize each effective donor update, avoiding one large adapter
            # dominating solely because of its displacement magnitude.
            normalized=s/s.norm()
            left.append((qB@u)*normalized)
            right.append((qA@vh.T)*normalized)
        u,sl,_=torch.linalg.svd(torch.cat(left,dim=1),full_matrices=False)
        v,sr,_=torch.linalg.svd(torch.cat(right,dim=1),full_matrices=False)
        d=CFG['basis_dim']
        assert min(len(sl),len(sr))>=d
        U,V=u[:,:d].contiguous(),v[:,:d].contiguous()
        assert torch.allclose(U.T@U,torch.eye(d),atol=2e-5)
        assert torch.allclose(V.T@V,torch.eye(d),atol=2e-5)
        bank[name]={'U':U,'V':V}
        diagnostics.append(dict(bank=label,layer=name,left_retained_energy=float(sl[:d].square().sum()/sl.square().sum()),
            right_retained_energy=float(sr[:d].square().sum()/sr.square().sum())))
    folder=ROOT/'bases';folder.mkdir(exist_ok=True)
    atomic_torch_save(bank,folder/(label+'.pt'))
    pd.DataFrame(diagnostics).to_csv(folder/(label+'_energy.csv'),index=False)
    write_json(folder/(label+'_sources.json'),dict(label=label,source_runs=[str(p.relative_to(ROOT)) for p in folders],
        extraction='Frobenius-normalized update energy, independently pooled left and right singular subspaces',basis_dim=CFG['basis_dim']))
    return bank

def extract_robust_basis(folders,label):
    """IRLS rank-d shared mean of effective updates, using low-rank factors.
    The robust p=1 objective is approximated for a fixed number of iterations.
    Store orthonormal singular bases; only target coefficients are optimized."""
    states=[load_result(p)['adapter'] for p in folders]
    bank={};diagnostics=[]
    for name in RUNNER.names:
        pairs=[(s[name]['A'].float(),2*s[name]['B'].float()) for s in states]
        norms=torch.tensor([float(((B.T@B)*(A@A.T).T).sum()) for A,B in pairs]).clamp_min(0)
        eps=max(float(norms.sqrt().median())*1e-3,1e-8)
        weights=torch.ones(len(pairs))/len(pairs)
        for iteration in range(CFG['irls_iterations']):
            weights=weights/weights.sum()
            L=torch.cat([B*weights[i].sqrt() for i,(A,B) in enumerate(pairs)],dim=1)
            R=torch.cat([A.T*weights[i].sqrt() for i,(A,B) in enumerate(pairs)],dim=1)
            qL,rL=torch.linalg.qr(L,mode='reduced')
            qR,rR=torch.linalg.qr(R,mode='reduced')
            u,sv,vh=torch.linalg.svd(rL@rR.T,full_matrices=False)
            d=CFG['basis_dim'];s=sv[:d];U=qL@u[:,:d];V=qR@vh[:d].T
            residuals=[]
            for i,(A,B) in enumerate(pairs):
                cross=((U.T@B)@(A@V)).diagonal().mul(s).sum()
                residuals.append((norms[i]+s.square().sum()-2*cross).clamp_min(0))
            residuals=torch.stack(residuals)
            if iteration+1<CFG['irls_iterations']:
                weights=1/torch.sqrt(residuals+eps**2)
        assert torch.isfinite(U).all() and torch.isfinite(V).all()
        assert torch.allclose(U.T@U,torch.eye(d),atol=2e-5)
        assert torch.allclose(V.T@V,torch.eye(d),atol=2e-5)
        bank[name]={'U':U.contiguous(),'V':V.contiguous()}
        for i in range(len(pairs)):
            diagnostics.append(dict(bank=label,layer=name,donor_index=i,weight=float(weights[i]/weights.sum()),
                                    residual_norm=float(residuals[i].sqrt())))
    folder=ROOT/'bases';folder.mkdir(exist_ok=True)
    atomic_torch_save(bank,folder/(label+'.pt'))
    pd.DataFrame(diagnostics).to_csv(folder/(label+'_irls.csv'),index=False)
    write_json(folder/(label+'_sources.json'),dict(label=label,source_runs=[str(p.relative_to(ROOT)) for p in folders],
        extraction='Rank-d weighted shared update mean with p=1 IRLS; orthonormal left/right singular bases',
        iterations=CFG['irls_iterations'],basis_dim=CFG['basis_dim']))
    return bank

def train_donors_and_bases():
    synthetic={attribute:[] for attribute in CFG['synthetic_attributes']}
    for task in SYNTH_TASKS:
        check_budget()
        folder=experiment('synthetic_donor',task['frames'][0],'donor',CFG['donor_lr'],
            CFG['donor_steps'],points=500,donor_group=task['frames'])
        synthetic[task['attribute']].append(folder)
    for attribute,folders in synthetic.items():
        BANKS[attribute]=extract_robust_basis(folders,attribute)
    all_synthetic=[p for folders in synthetic.values() for p in folders]
    BANKS['combined']=extract_robust_basis(all_synthetic,'combined')
    BANKS['combined_energy']=extract_energy_basis(all_synthetic,'combined_energy')
    real_folders=[]
    for scene in CFG['split']['donor']:
        group=[r for r in records_for('donor') if r['scene']==scene]
        real_folders.append(experiment('real_donor',group[0],'donor',CFG['donor_lr'],
            CFG['donor_steps'],points=100,donor_group=group))
    BANKS['shared_real']=extract_robust_basis(real_folders,'shared_real')
    # Fixed/hybrid zero-output checks, without using target GT.
    rec=records_for('validation')[0]
    rgb,_,_=load_frame(rec)
    with torch.no_grad():
        image=RUNNER.preprocess(rgb)
        RUNNER.install('frozen',47);reference=RUNNER.predict(image,tuple(rgb.shape[-2:])).cpu()
        for method in ['learned','hybrid','random']:
            RUNNER.install(method,47,BANKS['combined'] if method!='random' else None)
            output=RUNNER.predict(image,tuple(rgb.shape[-2:])).cpu()
            assert torch.allclose(reference,output,atol=1e-4,rtol=1e-3), 'Zero-output invariant failed: '+method
    RUNNER.clear()
    STAGE_STATUS['donors']='complete'
    journal('bases_ready',banks=list(BANKS))

try:
    train_donors_and_bases()
except BudgetStop as exc:
    STAGE_STATUS['donors']='budget_stopped';print(exc)
except Exception:
    STAGE_STATUS['donors']='failed';backup();raise
finally:
    write_json(ROOT/'stage_status.json',STAGE_STATUS)
backup()


## 9. Select learning rates on validation data only

In [ ]:
(ROOT / "code" / '08_validation.py').write_text("SELECTED={}\nMETHODS=['lora4','lora1','learned','random','hybrid','shared_real','learned_energy']\n\ndef bank_for(method):\n    if method=='learned_energy':return BANKS['combined_energy'],'combined_energy'\n    if method=='shared_real':return BANKS['shared_real'],'shared_real'\n    if method in ['learned','hybrid']:return BANKS['combined'],'combined'\n    return None,None\n\ndef final_row(folder):\n    return pd.read_csv(Path(folder)/'history.csv').sort_values('updates').iloc[-1]\n\ndef validate():\n    assert all(k in BANKS for k in ['combined','shared_real']), 'Finish donor/basis stage first.'\n    frozen=ROOT/'selected_settings.json'\n    if frozen.exists():\n        payload=json.loads(frozen.read_text())\n        assert payload['config_hash']==CONFIG_HASH\n        SELECTED.update(payload['selected'])\n        print('Reusing frozen validation selections:',SELECTED)\n        return\n    candidates=[]\n    for method in METHODS:\n        bank,label=bank_for(method)\n        for lr in CFG['lr_candidates'][method]:\n            values=[]\n            for rec in records_for('validation'):\n                p=experiment('validation',rec,method,lr,CFG['validation_steps'],\n                             bank=bank,bank_name=label)\n                values.append(float(final_row(p)['unobserved_absrel_pct']))\n            candidates.append(dict(method=method,lr=lr,mean_validation_absrel_pct=float(np.mean(values)),frames=len(values)))\n    table=pd.DataFrame(candidates)\n    table.to_csv(ROOT/'validation_candidates.csv',index=False)\n    for method in METHODS:\n        winner=table[table.method==method].sort_values(['mean_validation_absrel_pct','lr']).iloc[0]\n        SELECTED[method]=float(winner.lr)\n    write_json(frozen,dict(config_hash=CONFIG_HASH,selected=SELECTED,\n        selection='mean final unobserved AbsRel on validation frames only; all target evaluation settings frozen afterward',\n        validation_steps=CFG['validation_steps'],test_steps=CFG['test_steps']))\n    journal('validation_settings_frozen',selected=SELECTED)\n    print(table.to_string(index=False));print('FROZEN SETTINGS:',SELECTED)\n\nif STAGE_STATUS.get('donors')=='complete':\n    try:\n        validate();STAGE_STATUS['validation']='complete'\n    except BudgetStop as exc:\n        STAGE_STATUS['validation']='budget_stopped';print(exc)\n    except Exception:\n        STAGE_STATUS['validation']='failed';backup();raise\nelse:print('Validation not run: donor stage incomplete.')\nwrite_json(ROOT/'stage_status.json',STAGE_STATUS)\nbackup()\n", encoding="utf-8")
SELECTED={}
METHODS=['lora4','lora1','learned','random','hybrid','shared_real','learned_energy']

def bank_for(method):
    if method=='learned_energy':return BANKS['combined_energy'],'combined_energy'
    if method=='shared_real':return BANKS['shared_real'],'shared_real'
    if method in ['learned','hybrid']:return BANKS['combined'],'combined'
    return None,None

def final_row(folder):
    return pd.read_csv(Path(folder)/'history.csv').sort_values('updates').iloc[-1]

def validate():
    assert all(k in BANKS for k in ['combined','shared_real']), 'Finish donor/basis stage first.'
    frozen=ROOT/'selected_settings.json'
    if frozen.exists():
        payload=json.loads(frozen.read_text())
        assert payload['config_hash']==CONFIG_HASH
        SELECTED.update(payload['selected'])
        print('Reusing frozen validation selections:',SELECTED)
        return
    candidates=[]
    for method in METHODS:
        bank,label=bank_for(method)
        for lr in CFG['lr_candidates'][method]:
            values=[]
            for rec in records_for('validation'):
                p=experiment('validation',rec,method,lr,CFG['validation_steps'],
                             bank=bank,bank_name=label)
                values.append(float(final_row(p)['unobserved_absrel_pct']))
            candidates.append(dict(method=method,lr=lr,mean_validation_absrel_pct=float(np.mean(values)),frames=len(values)))
    table=pd.DataFrame(candidates)
    table.to_csv(ROOT/'validation_candidates.csv',index=False)
    for method in METHODS:
        winner=table[table.method==method].sort_values(['mean_validation_absrel_pct','lr']).iloc[0]
        SELECTED[method]=float(winner.lr)
    write_json(frozen,dict(config_hash=CONFIG_HASH,selected=SELECTED,
        selection='mean final unobserved AbsRel on validation frames only; all target evaluation settings frozen afterward',
        validation_steps=CFG['validation_steps'],test_steps=CFG['test_steps']))
    journal('validation_settings_frozen',selected=SELECTED)
    print(table.to_string(index=False));print('FROZEN SETTINGS:',SELECTED)

if STAGE_STATUS.get('donors')=='complete':
    try:
        validate();STAGE_STATUS['validation']='complete'
    except BudgetStop as exc:
        STAGE_STATUS['validation']='budget_stopped';print(exc)
    except Exception:
        STAGE_STATUS['validation']='failed';backup();raise
else:print('Validation not run: donor stage incomplete.')
write_json(ROOT/'stage_status.json',STAGE_STATUS)
backup()


## 10. Individual-attribute ablation and overlap diagnostic

In [ ]:
(ROOT / "code" / '11_attributes.py').write_text("# Diagnose individual synthetic attribute bases on VALIDATION ONLY. This does\n# not retune or select the combined basis using test outcomes. Basis dimensions,\n# coefficient LR and update budget are identical across these ablations.\ndef attribute_ablation():\n    rows=[]\n    for label in CFG['synthetic_attributes']+['combined']:\n        for rec in records_for('validation'):\n            p=experiment('attribute_validation',rec,'learned',SELECTED['learned'],CFG['validation_steps'],\n                         bank=BANKS[label],bank_name=label)\n            row=final_row(p)\n            rows.append(dict(attribute_basis=label,scene=rec['scene'],sample=rec['id'],\n                unobserved_absrel_pct=float(row['unobserved_absrel_pct']),\n                sparse_L1=float(row['sparse_L1'])))\n    table=pd.DataFrame(rows);table.to_csv(ROOT/'attribute_validation.csv',index=False)\n    print(table.groupby('attribute_basis').unobserved_absrel_pct.mean())\n    # Descriptive row/column basis overlaps; not a proof of semantic disentanglement.\n    overlaps=[]\n    attributes=CFG['synthetic_attributes']\n    for a in attributes:\n        for b in attributes:\n            lu=[];rv=[]\n            for name in RUNNER.names:\n                lu.append(float((BANKS[a][name]['U'].T@BANKS[b][name]['U']).square().sum()/CFG['basis_dim']))\n                rv.append(float((BANKS[a][name]['V'].T@BANKS[b][name]['V']).square().sum()/CFG['basis_dim']))\n            overlaps.append(dict(attribute_a=a,attribute_b=b,left_overlap=np.mean(lu),right_overlap=np.mean(rv)))\n    pd.DataFrame(overlaps).to_csv(ROOT/'attribute_basis_overlap.csv',index=False)\n\nif CFG['run_appearance_ablation'] and STAGE_STATUS.get('validation')=='complete':\n    try:\n        attribute_ablation();STAGE_STATUS['attribute_ablation']='complete'\n    except BudgetStop as exc:\n        STAGE_STATUS['attribute_ablation']='budget_stopped';print(exc)\n    except Exception:\n        STAGE_STATUS['attribute_ablation']='failed';backup();raise\nelse:print('Attribute ablation skipped: disabled or validation incomplete.')\nwrite_json(ROOT/'stage_status.json',STAGE_STATUS)\nbackup()\n", encoding="utf-8")
# Diagnose individual synthetic attribute bases on VALIDATION ONLY. This does
# not retune or select the combined basis using test outcomes. Basis dimensions,
# coefficient LR and update budget are identical across these ablations.
def attribute_ablation():
    rows=[]
    for label in CFG['synthetic_attributes']+['combined']:
        for rec in records_for('validation'):
            p=experiment('attribute_validation',rec,'learned',SELECTED['learned'],CFG['validation_steps'],
                         bank=BANKS[label],bank_name=label)
            row=final_row(p)
            rows.append(dict(attribute_basis=label,scene=rec['scene'],sample=rec['id'],
                unobserved_absrel_pct=float(row['unobserved_absrel_pct']),
                sparse_L1=float(row['sparse_L1'])))
    table=pd.DataFrame(rows);table.to_csv(ROOT/'attribute_validation.csv',index=False)
    print(table.groupby('attribute_basis').unobserved_absrel_pct.mean())
    # Descriptive row/column basis overlaps; not a proof of semantic disentanglement.
    overlaps=[]
    attributes=CFG['synthetic_attributes']
    for a in attributes:
        for b in attributes:
            lu=[];rv=[]
            for name in RUNNER.names:
                lu.append(float((BANKS[a][name]['U'].T@BANKS[b][name]['U']).square().sum()/CFG['basis_dim']))
                rv.append(float((BANKS[a][name]['V'].T@BANKS[b][name]['V']).square().sum()/CFG['basis_dim']))
            overlaps.append(dict(attribute_a=a,attribute_b=b,left_overlap=np.mean(lu),right_overlap=np.mean(rv)))
    pd.DataFrame(overlaps).to_csv(ROOT/'attribute_basis_overlap.csv',index=False)

if CFG['run_appearance_ablation'] and STAGE_STATUS.get('validation')=='complete':
    try:
        attribute_ablation();STAGE_STATUS['attribute_ablation']='complete'
    except BudgetStop as exc:
        STAGE_STATUS['attribute_ablation']='budget_stopped';print(exc)
    except Exception:
        STAGE_STATUS['attribute_ablation']='failed';backup();raise
else:print('Attribute ablation skipped: disabled or validation incomplete.')
write_json(ROOT/'stage_status.json',STAGE_STATUS)
backup()


## 11. Main test: ordinary, random, learned and hybrid adaptation

In [ ]:
(ROOT / "code" / '09_main.py').write_text("def run_main():\n    assert SELECTED,'Freeze validation choices first.'\n    for rec in records_for('test'):\n        # Establish a common aligned base prediction for this measurement set.\n        experiment('test',rec,'frozen',0.,0)\n    # Run one complete seed across all methods/scenes before extra seeds.\n    # If the budget stops, the report states exactly which runs are missing.\n    for seed in CFG['main_seeds']:\n        for method in METHODS:\n            bank,label=bank_for(method)\n            for rec in records_for('test'):\n                experiment('test',rec,method,SELECTED[method],CFG['test_steps'],seed=seed,\n                           bank=bank,bank_name=label)\n\nif STAGE_STATUS.get('validation')=='complete':\n    try:\n        run_main();STAGE_STATUS['main']='complete'\n    except BudgetStop as exc:\n        STAGE_STATUS['main']='budget_stopped';print(exc)\n    except Exception:\n        STAGE_STATUS['main']='failed';backup();raise\nelse:print('Main test not run: validation incomplete.')\nwrite_json(ROOT/'stage_status.json',STAGE_STATUS)\nbackup()\n", encoding="utf-8")
def run_main():
    assert SELECTED,'Freeze validation choices first.'
    for rec in records_for('test'):
        # Establish a common aligned base prediction for this measurement set.
        experiment('test',rec,'frozen',0.,0)
    # Run one complete seed across all methods/scenes before extra seeds.
    # If the budget stops, the report states exactly which runs are missing.
    for seed in CFG['main_seeds']:
        for method in METHODS:
            bank,label=bank_for(method)
            for rec in records_for('test'):
                experiment('test',rec,method,SELECTED[method],CFG['test_steps'],seed=seed,
                           bank=bank,bank_name=label)

if STAGE_STATUS.get('validation')=='complete':
    try:
        run_main();STAGE_STATUS['main']='complete'
    except BudgetStop as exc:
        STAGE_STATUS['main']='budget_stopped';print(exc)
    except Exception:
        STAGE_STATUS['main']='failed';backup();raise
else:print('Main test not run: validation incomplete.')
write_json(ROOT/'stage_status.json',STAGE_STATUS)
backup()


## 12. Sparse-point and outlier experiments

In [ ]:
(ROOT / "code" / '10_robustness.py').write_text("# Sparse-point and noise studies use the FIRST selected frame from EACH test\n# scene, a common 30-update budget and fixed settings. No test retuning.\ndef first_test_frames():\n    return [next(r for r in records_for('test') if r['scene']==scene) for scene in CFG['split']['test']]\n\ndef run_robustness():\n    conditions=[(100,0.),(20,0.),(500,0.),(100,.1),(100,.2)]\n    for points,noise in conditions:\n        for rec in first_test_frames():\n            experiment('robustness',rec,'frozen',0.,0,points=points,noise=noise)\n        for method in ['lora4','learned','random','hybrid']:\n            bank,label=bank_for(method)\n            for rec in first_test_frames():\n                experiment('robustness',rec,method,SELECTED[method],CFG['robustness_steps'],\n                    points=points,noise=noise,bank=bank,bank_name=label)\n\nif CFG['run_robustness'] and STAGE_STATUS.get('main')=='complete':\n    try:\n        run_robustness();STAGE_STATUS['robustness']='complete'\n    except BudgetStop as exc:\n        STAGE_STATUS['robustness']='budget_stopped';print(exc)\n    except Exception:\n        STAGE_STATUS['robustness']='failed';backup();raise\nelse:print('Robustness skipped: disabled or main stage incomplete.')\nwrite_json(ROOT/'stage_status.json',STAGE_STATUS)\nbackup()\n", encoding="utf-8")
# Sparse-point and noise studies use the FIRST selected frame from EACH test
# scene, a common 30-update budget and fixed settings. No test retuning.
def first_test_frames():
    return [next(r for r in records_for('test') if r['scene']==scene) for scene in CFG['split']['test']]

def run_robustness():
    conditions=[(100,0.),(20,0.),(500,0.),(100,.1),(100,.2)]
    for points,noise in conditions:
        for rec in first_test_frames():
            experiment('robustness',rec,'frozen',0.,0,points=points,noise=noise)
        for method in ['lora4','learned','random','hybrid']:
            bank,label=bank_for(method)
            for rec in first_test_frames():
                experiment('robustness',rec,method,SELECTED[method],CFG['robustness_steps'],
                    points=points,noise=noise,bank=bank,bank_name=label)

if CFG['run_robustness'] and STAGE_STATUS.get('main')=='complete':
    try:
        run_robustness();STAGE_STATUS['robustness']='complete'
    except BudgetStop as exc:
        STAGE_STATUS['robustness']='budget_stopped';print(exc)
    except Exception:
        STAGE_STATUS['robustness']='failed';backup();raise
else:print('Robustness skipped: disabled or main stage incomplete.')
write_json(ROOT/'stage_status.json',STAGE_STATUS)
backup()


## 13. Generate measured-result tables, figures, report and backup

In [ ]:
(ROOT / "code" / '12_report.py').write_text('# Always run this cell, including after a compute-budget stop. Only completed\n# runs enter result tables; missing/failed runs are listed rather than invented.\nREPORT=ROOT/\'report\'\nREPORT.mkdir(exist_ok=True)\nrecord_lookup={r[\'id\']:r for r in MANIFEST[\'frames\']} if \'MANIFEST\' in globals() else {}\nall_rows=[];failed=[]\nfor folder in sorted(RUNS.iterdir()):\n    if not folder.is_dir() or not (folder/\'settings.json\').exists():continue\n    settings=json.loads((folder/\'settings.json\').read_text())\n    if settings[\'config_hash\']!=CONFIG_HASH:continue\n    if not (folder/\'complete.json\').exists():\n        failed.append(dict(folder=folder.name,status=\'failed\' if (folder/\'failure.json\').exists() else \'incomplete\'))\n        continue\n    status=json.loads((folder/\'complete.json\').read_text())\n    h=pd.read_csv(folder/\'history.csv\').drop_duplicates(\'updates\',keep=\'last\')\n    rec=record_lookup.get(settings[\'sample\'],{})\n    for _,row in h.iterrows():\n        all_rows.append(dict(**row.to_dict(),stage=settings[\'stage\'],sample=settings[\'sample\'],\n            scene=rec.get(\'scene\',settings[\'sample\']),method=settings[\'method\'],lr=settings[\'lr\'],\n            seed=settings[\'seed\'],points=settings[\'points\'],noise=settings[\'noise\'],\n            bank_name=settings[\'bank_name\'],folder=str(folder.relative_to(ROOT)),\n            peak_memory_GB=status.get(\'peak_memory_GB\',np.nan)))\nALL=pd.DataFrame(all_rows)\nALL.to_csv(REPORT/\'all_completed_measurements.csv\',index=False)\nwrite_json(REPORT/\'incomplete_runs.json\',failed)\nprint(\'Completed runs:\',len(list(RUNS.glob(\'*/complete.json\'))),\'incomplete/failed:\',len(failed))\n\ntable_text=[]\nif len(ALL):\n    FINALS=ALL.sort_values(\'updates\').groupby(\'folder\',as_index=False).tail(1)\n    FINALS.to_csv(REPORT/\'final_results.csv\',index=False)\n    main=FINALS[FINALS.stage==\'test\'].copy()\n    if len(main):\n        expected_main=len(records_for(\'test\'))*(1+len(METHODS)*len(CFG[\'main_seeds\']))\n        coverage=dict(completed_runs=len(main),expected_runs=expected_main,\n                      complete=len(main)==expected_main)\n        write_json(REPORT/\'main_coverage.json\',coverage)\n        if not coverage[\'complete\']:\n            print(\'PARTIAL MAIN STUDY: do not rank unbalanced method averages.\',coverage)\n        table_text.append((\'Main coverage (unbalanced partial averages must not be ranked)\',json.dumps(coverage,indent=2)))\n        # Average frames within sequence before aggregating sequences. Seeds\n        # describe initialization sensitivity, not additional independent scenes.\n        scene_seed=main.groupby([\'method\',\'scene\',\'seed\'],as_index=False).agg(\n            unobserved_absrel_pct=(\'unobserved_absrel_pct\',\'mean\'),all_absrel_pct=(\'all_absrel_pct\',\'mean\'),\n            observed_absrel_pct=(\'observed_absrel_pct\',\'mean\'),unobserved_rmse=(\'unobserved_rmse\',\'mean\'),\n            adaptation_seconds=(\'adaptation_seconds\',\'mean\'),trainable_parameters=(\'trainable_parameters\',\'first\'),\n            frames=(\'sample\',\'nunique\'),peak_memory_GB=(\'peak_memory_GB\',\'max\'))\n        scene_seed.to_csv(REPORT/\'main_scene_seed.csv\',index=False)\n        scene_mean=scene_seed.groupby([\'method\',\'scene\'],as_index=False).agg(\n            unobserved_absrel_pct=(\'unobserved_absrel_pct\',\'mean\'),seed_sd=(\'unobserved_absrel_pct\',\'std\'),\n            all_absrel_pct=(\'all_absrel_pct\',\'mean\'),observed_absrel_pct=(\'observed_absrel_pct\',\'mean\'),\n            adaptation_seconds=(\'adaptation_seconds\',\'mean\'),trainable_parameters=(\'trainable_parameters\',\'first\'),\n            seeds=(\'seed\',\'nunique\'),frames=(\'frames\',\'min\'),peak_memory_GB=(\'peak_memory_GB\',\'max\'))\n        scene_mean.to_csv(REPORT/\'main_scene_means.csv\',index=False)\n        summary_main=scene_mean.groupby(\'method\').agg(\n            mean_scene_absrel_pct=(\'unobserved_absrel_pct\',\'mean\'),scene_sd=(\'unobserved_absrel_pct\',\'std\'),\n            scenes=(\'scene\',\'nunique\'),min_seeds=(\'seeds\',\'min\'),min_frames_per_scene=(\'frames\',\'min\'),\n            mean_adaptation_seconds=(\'adaptation_seconds\',\'mean\'),trainable_parameters=(\'trainable_parameters\',\'first\'),\n            peak_memory_GB=(\'peak_memory_GB\',\'max\'))\n        summary_main.to_csv(REPORT/\'main_summary.csv\')\n        print(\'\\nMAIN SUMMARY (check coverage columns):\\n\',summary_main.to_string())\n        table_text.append((\'Main test summary\',summary_main.to_string()))\n        fig,axs=plt.subplots(1,2,figsize=(12,4))\n        histories=ALL[ALL.stage==\'test\']\n        for method in METHODS if \'METHODS\' in globals() else sorted(main.method.unique()):\n            a=histories[histories.method==method]\n            if not len(a):continue\n            # Equal frames were selected per scene; curves are descriptive means.\n            curve=a.groupby([\'scene\',\'updates\'],as_index=False)[[\'unobserved_absrel_pct\',\'sparse_L1\']].mean()\n            curve=curve.groupby(\'updates\')[[\'unobserved_absrel_pct\',\'sparse_L1\']].mean()\n            axs[0].plot(curve.index,curve.unobserved_absrel_pct,label=method)\n            axs[1].plot(curve.index,curve.sparse_L1,label=method)\n        base=main[main.method==\'frozen\']\n        if len(base):axs[0].axhline(base.unobserved_absrel_pct.mean(),ls=\':\',color=\'black\',label=\'aligned base\')\n        for ax in axs:ax.set_xlabel(\'Successful updates\');ax.grid(alpha=.2);ax.legend(fontsize=8)\n        axs[0].set_ylabel(\'Unobserved AbsRel (%)\');axs[1].set_ylabel(\'Supplied-depth L1\')\n        fig.suptitle(\'Single-frame ScanNet pilot: completed runs only; see coverage tables\')\n        fig.tight_layout()\n        for ext in [\'png\',\'pdf\']:fig.savefig(REPORT/f\'main_convergence.{ext}\',dpi=180)\n        display(fig);plt.close(fig)\n        # Predeclared descriptive target: 25% reduction from aligned-base error.\n        targets=[]\n        for sample in base[\'sample\'].unique():\n            threshold=.75*float(base[base[\'sample\']==sample].unobserved_absrel_pct.iloc[0])\n            for folder,a in histories[(histories[\'sample\']==sample)&(histories.method!=\'frozen\')].groupby(\'folder\'):\n                a=a.sort_values(\'updates\');reached=a[a.unobserved_absrel_pct<=threshold]\n                first=reached.iloc[0] if len(reached) else None\n                targets.append(dict(sample=sample,method=a.method.iloc[0],seed=int(a.seed.iloc[0]),\n                    target_absrel_pct=threshold,reached=first is not None,\n                    first_recorded_update=float(first.updates) if first is not None else np.nan,\n                    time_to_prediction_seconds=float(first.time_to_prediction_seconds) if first is not None else np.nan))\n        pd.DataFrame(targets).to_csv(REPORT/\'time_to_25pct_improvement.csv\',index=False)\n        # A common-scale qualitative comparison for the first test frame.\n        first_sample=next((r[\'id\'] for r in records_for(\'test\') if r[\'id\'] in set(main[\'sample\'])),None)\n        if first_sample:\n            rec=record_lookup[first_sample];rgb,gt,valid=load_frame(rec)\n            subset=main[(main[\'sample\']==first_sample)&(main.seed==47)]\n            picks=[\'frozen\',\'lora4\',\'learned\',\'random\',\'hybrid\']\n            predictions=[]\n            for method in picks:\n                a=subset[subset.method==method]\n                if len(a):\n                    result=torch.load(ROOT/a.iloc[0].folder/\'result.pt\',map_location=\'cpu\',weights_only=False)\n                    predictions.append((method,result[\'prediction\'][0].numpy()))\n            if predictions:\n                truth=gt[0].numpy();vm=valid[0].numpy()\n                vmax=float(np.percentile(truth[vm],98))\n                errors=[np.where(vm,np.abs(p-truth),np.nan) for _,p in predictions]\n                emax=max(float(np.nanpercentile(np.stack(errors),95)),1e-4)\n                fig,axs=plt.subplots(2,len(predictions),figsize=(3*len(predictions),6),squeeze=False)\n                for col,((method,p),e) in enumerate(zip(predictions,errors)):\n                    axs[0,col].imshow(p,cmap=\'viridis\',vmin=0,vmax=vmax);axs[0,col].set_title(method)\n                    axs[1,col].imshow(e,cmap=\'magma\',vmin=0,vmax=emax)\n                for ax in axs.flat:ax.axis(\'off\')\n                fig.suptitle(first_sample+\' | common depth and absolute-error color scales\')\n                fig.tight_layout();fig.savefig(REPORT/\'qualitative.png\',dpi=180);plt.close(fig)\n    robustness=FINALS[FINALS.stage==\'robustness\']\n    if len(robustness):\n        robust_scene=robustness.groupby([\'method\',\'scene\',\'points\',\'noise\'],as_index=False).unobserved_absrel_pct.mean()\n        robust_summary=robust_scene.groupby([\'method\',\'points\',\'noise\']).agg(\n            mean_absrel_pct=(\'unobserved_absrel_pct\',\'mean\'),scenes=(\'scene\',\'nunique\'))\n        robust_summary.to_csv(REPORT/\'robustness_summary.csv\')\n        table_text.append((\'Robustness (30-update budget; partial coverage explicitly listed)\',robust_summary.to_string()))\n        fig,axs=plt.subplots(1,2,figsize=(12,4))\n        for method,a in robust_scene.groupby(\'method\'):\n            clean=a[a.noise==0].groupby(\'points\').unobserved_absrel_pct.mean()\n            noisy=a[a.points==100].groupby(\'noise\').unobserved_absrel_pct.mean()\n            if len(clean):axs[0].plot(clean.index,clean.values,marker=\'o\',label=method)\n            if len(noisy):axs[1].plot(100*noisy.index,noisy.values,marker=\'o\',label=method)\n        axs[0].set_xlabel(\'Supplied depth points\');axs[0].set_xscale(\'log\')\n        axs[1].set_xlabel(\'Corrupted conditioning points (%)\')\n        for ax in axs:ax.set_ylabel(\'Unobserved AbsRel (%)\');ax.grid(alpha=.2);ax.legend(fontsize=8)\n        fig.tight_layout()\n        for ext in [\'png\',\'pdf\']:fig.savefig(REPORT/f\'robustness.{ext}\',dpi=180)\n        display(fig);plt.close(fig)\n\nif (ROOT/\'attribute_validation.csv\').exists():\n    attributes=pd.read_csv(ROOT/\'attribute_validation.csv\')\n    ab=attributes.groupby(\'attribute_basis\').unobserved_absrel_pct.mean()\n    table_text.append((\'Attribute bases: validation diagnostic only\',ab.to_string()))\n    fig,ax=plt.subplots(figsize=(8,4));ab.plot.bar(ax=ax)\n    ax.set_ylabel(\'Validation unobserved AbsRel (%)\');ax.set_xlabel(\'Basis source (equal dimensions)\')\n    fig.tight_layout();fig.savefig(REPORT/\'attribute_ablation.png\',dpi=180);plt.close(fig)\nif (ROOT/\'attribute_basis_overlap.csv\').exists():\n    ov=pd.read_csv(ROOT/\'attribute_basis_overlap.csv\')\n    fig,axs=plt.subplots(1,2,figsize=(10,4))\n    for ax,field in zip(axs,[\'left_overlap\',\'right_overlap\']):\n        matrix=ov.pivot(index=\'attribute_a\',columns=\'attribute_b\',values=field)\n        im=ax.imshow(matrix.values,vmin=0,vmax=1,cmap=\'viridis\')\n        ax.set_xticks(range(len(matrix)),matrix.columns,rotation=45,ha=\'right\')\n        ax.set_yticks(range(len(matrix)),matrix.index);ax.set_title(field)\n    fig.colorbar(im,ax=axs.tolist(),label=\'Normalized projection overlap\')\n    fig.savefig(REPORT/\'attribute_overlap.png\',dpi=180,bbox_inches=\'tight\');plt.close(fig)\n\nstatus=json.loads((ROOT/\'stage_status.json\').read_text()) if (ROOT/\'stage_status.json\').exists() else {}\ndescription=\'\'\'# DA4022 final evaluation: measured results\n\n## Objective\nInvestigate whether fixed bases derived from controlled 3D variations can guide sparse-depth test-time adaptation, and whether a small residual LoRA restores flexibility.\n\n## Data and separation\nThis is a small custom ScanNet single-frame adaptation study using the official CAPA sample tensors. Donor sequences: scene0707 and scene0766. Validation: scene0771. Test: scene0777 and scene0806. Two frames per sequence are used. No sequence crosses splits. The original sample conditioning is replaced by explicitly seeded random point measurements from valid dense GT. This is not the original CAPA sequence-level protocol or a standard zero-shot benchmark result.\n\n## Controlled synthetic attributes\nA simple analytic room/box/sphere renderer generates geometry, camera, texture, and illumination interventions. Within each adapter task, other scene parameters remain fixed and only the named attribute varies; one adapter is shared across the three variants. Two nuisance base scenes are used per attribute. This is low-diversity procedural data, not a reproduction of MegaSynth. Synthetic depth is known from the renderer. Test RGB/GT do not enter basis extraction.\n\n## Methods\nFrozen VGGT + sparse-depth alignment; ordinary image-encoder QKV LoRA rank 4; rank-1 LoRA as a near-budget comparator for the hybrid; random fixed bases; pooled synthetic attribute bases; real-donor bases; and synthetic bases + rank-1 residual LoRA. Main learned bases use a rank-d robust shared update mean, approximated by five p=1 IRLS iterations with low-rank QR/SVD factors. Orthonormal singular bases are retained for coefficient optimization. An additional learned_energy method pools Frobenius-normalized uncentered update energy as an extraction ablation. This is a small independent implementation inspired by the shared-update objective, not a full reproduction of the original paper\'s data and training pipeline. Four-dimensional left/right bases and a trainable 4x4 matrix per layer are used. Combined and individual attribute bases have equal dimensions, but combined bases use more donor tasks. LoRA and residual scaling are 2. All weight decay is zero. The coefficient and residual learning rates may differ; the exact choices are recorded.\n\n## Selection, masks, and timing\nMethod learning rates are selected using final unobserved-region error on validation frames at a common 30-update budget, then frozen. Main evaluation uses up to 100 updates. GT is used for metrics and generating the declared sparse measurement simulation; no other dense test supervision enters gradients or alignment. Noise multiplies a seeded subset of observed values by uniform[0.2,2.0] factors. All methods share conditioning and corruption for an image/condition. Invalid predictions cause failed runs rather than exclusion. Nonpositive predictions remain in error metrics and their fractions are recorded. No test-best checkpoint is used for final scores.\n\nTimings include synchronized forward/alignment/backward/optimizer work and exclude metrics, file saving, preprocessing and model loading. These are instrumented pilot timings with checkpointing and warm-cache effects, not deployment latency. Offline donor costs are available in the raw measurements. The reported time-to-target is the first recorded checkpoint reaching a predeclared 25% improvement over the aligned baseline; it is not time to a global optimum.\n\n## Interpretation and limits\nTables are generated only from complete runs. Check scene/frame/seed coverage before comparing aggregates. Frames/seeds are not independent scenes. Only two real donor scenes, one validation scene, two test scenes and two synthetic base scenes are used: no broad generalization or statistical significance claim is justified. Seeds vary LoRA initialization and random bases; pure fixed-basis coefficient runs have no random initialization. CUDA reproducibility is best-effort, and warnings/versions are recorded. Attribute overlap is descriptive and does not establish semantic disentanglement. The quadratic experiment deliberately supplies an optimum-containing subspace and demonstrates a mechanism rather than a VGGT guarantee. Historical mid-evaluation numbers are not mixed into these tables. Any accuracy, convergence or robustness benefit must be supported by the generated results, not assumed.\n\'\'\'\ndescription+=\'\\n## Stage completion\\n\\n```json\\n\'+json.dumps(status,indent=2)+\'\\n```\\n\'\ndescription+=\'\\nIncomplete/failed runs: \'+str(len(failed))+\' (see incomplete_runs.json).\\n\'\nfor title,text in table_text:\n    description+=\'\\n## \'+title+\'\\n\\n```text\\n\'+text+\'\\n```\\n\'\ndescription+=\'\'\'\n## Presentation sequence\n1. Proposal: supervision from CAPA, reusable directions from attribute mining.\n2. Controlled known-optimum experiment and its misalignment failure.\n3. Actual synthetic interventions, donor/validation/test separation, adapter equations.\n4. Validation choices and main test accuracy/parameter/convergence tables.\n5. Attribute ablation and learned-versus-random findings, including negative outcomes.\n6. Residual-adapter, sparsity and outlier results.\n7. Limitations, offline cost, failure cases and conclusions limited to the measured evidence.\n\n## Sources\n- CAPA code and license: https://github.com/nv-dvl/capa\n- VGGT: https://github.com/facebookresearch/vggt\n- Attribute-subspace paper: https://arxiv.org/abs/2604.10095\n- Original course proposal and historical findings are maintained separately in the local workspace.\n\'\'\'\n(REPORT/\'FINAL_REPORT.md\').write_text(description,encoding=\'utf-8\')\nimport html\nimages=\'\'.join(f\'<h2>{html.escape(p.stem)}</h2><img src="{p.name}" style="max-width:100%">\' for p in sorted(REPORT.glob(\'*.png\')))\n(REPORT/\'FINAL_REPORT.html\').write_text(\'<!doctype html><meta charset="utf-8"><title>DA4022 results</title>\'\n    \'<style>body{font:15px system-ui;max-width:1200px;margin:40px auto;padding:20px}pre{white-space:pre-wrap}</style>\'\n    \'<pre>\'+html.escape(description)+\'</pre>\'+images,encoding=\'utf-8\')\njournal(\'report_generated\',stage_status=status,completed_measurements=len(ALL),incomplete_runs=len(failed))\nprint(\'Reports:\',REPORT/\'FINAL_REPORT.md\',REPORT/\'FINAL_REPORT.html\')\nprint(\'Important: inspect generated tables. No method is assumed to win.\')\nbackup()\n', encoding="utf-8")
# Always run this cell, including after a compute-budget stop. Only completed
# runs enter result tables; missing/failed runs are listed rather than invented.
REPORT=ROOT/'report'
REPORT.mkdir(exist_ok=True)
record_lookup={r['id']:r for r in MANIFEST['frames']} if 'MANIFEST' in globals() else {}
all_rows=[];failed=[]
for folder in sorted(RUNS.iterdir()):
    if not folder.is_dir() or not (folder/'settings.json').exists():continue
    settings=json.loads((folder/'settings.json').read_text())
    if settings['config_hash']!=CONFIG_HASH:continue
    if not (folder/'complete.json').exists():
        failed.append(dict(folder=folder.name,status='failed' if (folder/'failure.json').exists() else 'incomplete'))
        continue
    status=json.loads((folder/'complete.json').read_text())
    h=pd.read_csv(folder/'history.csv').drop_duplicates('updates',keep='last')
    rec=record_lookup.get(settings['sample'],{})
    for _,row in h.iterrows():
        all_rows.append(dict(**row.to_dict(),stage=settings['stage'],sample=settings['sample'],
            scene=rec.get('scene',settings['sample']),method=settings['method'],lr=settings['lr'],
            seed=settings['seed'],points=settings['points'],noise=settings['noise'],
            bank_name=settings['bank_name'],folder=str(folder.relative_to(ROOT)),
            peak_memory_GB=status.get('peak_memory_GB',np.nan)))
ALL=pd.DataFrame(all_rows)
ALL.to_csv(REPORT/'all_completed_measurements.csv',index=False)
write_json(REPORT/'incomplete_runs.json',failed)
print('Completed runs:',len(list(RUNS.glob('*/complete.json'))),'incomplete/failed:',len(failed))

table_text=[]
if len(ALL):
    FINALS=ALL.sort_values('updates').groupby('folder',as_index=False).tail(1)
    FINALS.to_csv(REPORT/'final_results.csv',index=False)
    main=FINALS[FINALS.stage=='test'].copy()
    if len(main):
        expected_main=len(records_for('test'))*(1+len(METHODS)*len(CFG['main_seeds']))
        coverage=dict(completed_runs=len(main),expected_runs=expected_main,
                      complete=len(main)==expected_main)
        write_json(REPORT/'main_coverage.json',coverage)
        if not coverage['complete']:
            print('PARTIAL MAIN STUDY: do not rank unbalanced method averages.',coverage)
        table_text.append(('Main coverage (unbalanced partial averages must not be ranked)',json.dumps(coverage,indent=2)))
        # Average frames within sequence before aggregating sequences. Seeds
        # describe initialization sensitivity, not additional independent scenes.
        scene_seed=main.groupby(['method','scene','seed'],as_index=False).agg(
            unobserved_absrel_pct=('unobserved_absrel_pct','mean'),all_absrel_pct=('all_absrel_pct','mean'),
            observed_absrel_pct=('observed_absrel_pct','mean'),unobserved_rmse=('unobserved_rmse','mean'),
            adaptation_seconds=('adaptation_seconds','mean'),trainable_parameters=('trainable_parameters','first'),
            frames=('sample','nunique'),peak_memory_GB=('peak_memory_GB','max'))
        scene_seed.to_csv(REPORT/'main_scene_seed.csv',index=False)
        scene_mean=scene_seed.groupby(['method','scene'],as_index=False).agg(
            unobserved_absrel_pct=('unobserved_absrel_pct','mean'),seed_sd=('unobserved_absrel_pct','std'),
            all_absrel_pct=('all_absrel_pct','mean'),observed_absrel_pct=('observed_absrel_pct','mean'),
            adaptation_seconds=('adaptation_seconds','mean'),trainable_parameters=('trainable_parameters','first'),
            seeds=('seed','nunique'),frames=('frames','min'),peak_memory_GB=('peak_memory_GB','max'))
        scene_mean.to_csv(REPORT/'main_scene_means.csv',index=False)
        summary_main=scene_mean.groupby('method').agg(
            mean_scene_absrel_pct=('unobserved_absrel_pct','mean'),scene_sd=('unobserved_absrel_pct','std'),
            scenes=('scene','nunique'),min_seeds=('seeds','min'),min_frames_per_scene=('frames','min'),
            mean_adaptation_seconds=('adaptation_seconds','mean'),trainable_parameters=('trainable_parameters','first'),
            peak_memory_GB=('peak_memory_GB','max'))
        summary_main.to_csv(REPORT/'main_summary.csv')
        print('\nMAIN SUMMARY (check coverage columns):\n',summary_main.to_string())
        table_text.append(('Main test summary',summary_main.to_string()))
        fig,axs=plt.subplots(1,2,figsize=(12,4))
        histories=ALL[ALL.stage=='test']
        for method in METHODS if 'METHODS' in globals() else sorted(main.method.unique()):
            a=histories[histories.method==method]
            if not len(a):continue
            # Equal frames were selected per scene; curves are descriptive means.
            curve=a.groupby(['scene','updates'],as_index=False)[['unobserved_absrel_pct','sparse_L1']].mean()
            curve=curve.groupby('updates')[['unobserved_absrel_pct','sparse_L1']].mean()
            axs[0].plot(curve.index,curve.unobserved_absrel_pct,label=method)
            axs[1].plot(curve.index,curve.sparse_L1,label=method)
        base=main[main.method=='frozen']
        if len(base):axs[0].axhline(base.unobserved_absrel_pct.mean(),ls=':',color='black',label='aligned base')
        for ax in axs:ax.set_xlabel('Successful updates');ax.grid(alpha=.2);ax.legend(fontsize=8)
        axs[0].set_ylabel('Unobserved AbsRel (%)');axs[1].set_ylabel('Supplied-depth L1')
        fig.suptitle('Single-frame ScanNet pilot: completed runs only; see coverage tables')
        fig.tight_layout()
        for ext in ['png','pdf']:fig.savefig(REPORT/f'main_convergence.{ext}',dpi=180)
        display(fig);plt.close(fig)
        # Predeclared descriptive target: 25% reduction from aligned-base error.
        targets=[]
        for sample in base['sample'].unique():
            threshold=.75*float(base[base['sample']==sample].unobserved_absrel_pct.iloc[0])
            for folder,a in histories[(histories['sample']==sample)&(histories.method!='frozen')].groupby('folder'):
                a=a.sort_values('updates');reached=a[a.unobserved_absrel_pct<=threshold]
                first=reached.iloc[0] if len(reached) else None
                targets.append(dict(sample=sample,method=a.method.iloc[0],seed=int(a.seed.iloc[0]),
                    target_absrel_pct=threshold,reached=first is not None,
                    first_recorded_update=float(first.updates) if first is not None else np.nan,
                    time_to_prediction_seconds=float(first.time_to_prediction_seconds) if first is not None else np.nan))
        pd.DataFrame(targets).to_csv(REPORT/'time_to_25pct_improvement.csv',index=False)
        # A common-scale qualitative comparison for the first test frame.
        first_sample=next((r['id'] for r in records_for('test') if r['id'] in set(main['sample'])),None)
        if first_sample:
            rec=record_lookup[first_sample];rgb,gt,valid=load_frame(rec)
            subset=main[(main['sample']==first_sample)&(main.seed==47)]
            picks=['frozen','lora4','learned','random','hybrid']
            predictions=[]
            for method in picks:
                a=subset[subset.method==method]
                if len(a):
                    result=torch.load(ROOT/a.iloc[0].folder/'result.pt',map_location='cpu',weights_only=False)
                    predictions.append((method,result['prediction'][0].numpy()))
            if predictions:
                truth=gt[0].numpy();vm=valid[0].numpy()
                vmax=float(np.percentile(truth[vm],98))
                errors=[np.where(vm,np.abs(p-truth),np.nan) for _,p in predictions]
                emax=max(float(np.nanpercentile(np.stack(errors),95)),1e-4)
                fig,axs=plt.subplots(2,len(predictions),figsize=(3*len(predictions),6),squeeze=False)
                for col,((method,p),e) in enumerate(zip(predictions,errors)):
                    axs[0,col].imshow(p,cmap='viridis',vmin=0,vmax=vmax);axs[0,col].set_title(method)
                    axs[1,col].imshow(e,cmap='magma',vmin=0,vmax=emax)
                for ax in axs.flat:ax.axis('off')
                fig.suptitle(first_sample+' | common depth and absolute-error color scales')
                fig.tight_layout();fig.savefig(REPORT/'qualitative.png',dpi=180);plt.close(fig)
    robustness=FINALS[FINALS.stage=='robustness']
    if len(robustness):
        robust_scene=robustness.groupby(['method','scene','points','noise'],as_index=False).unobserved_absrel_pct.mean()
        robust_summary=robust_scene.groupby(['method','points','noise']).agg(
            mean_absrel_pct=('unobserved_absrel_pct','mean'),scenes=('scene','nunique'))
        robust_summary.to_csv(REPORT/'robustness_summary.csv')
        table_text.append(('Robustness (30-update budget; partial coverage explicitly listed)',robust_summary.to_string()))
        fig,axs=plt.subplots(1,2,figsize=(12,4))
        for method,a in robust_scene.groupby('method'):
            clean=a[a.noise==0].groupby('points').unobserved_absrel_pct.mean()
            noisy=a[a.points==100].groupby('noise').unobserved_absrel_pct.mean()
            if len(clean):axs[0].plot(clean.index,clean.values,marker='o',label=method)
            if len(noisy):axs[1].plot(100*noisy.index,noisy.values,marker='o',label=method)
        axs[0].set_xlabel('Supplied depth points');axs[0].set_xscale('log')
        axs[1].set_xlabel('Corrupted conditioning points (%)')
        for ax in axs:ax.set_ylabel('Unobserved AbsRel (%)');ax.grid(alpha=.2);ax.legend(fontsize=8)
        fig.tight_layout()
        for ext in ['png','pdf']:fig.savefig(REPORT/f'robustness.{ext}',dpi=180)
        display(fig);plt.close(fig)

if (ROOT/'attribute_validation.csv').exists():
    attributes=pd.read_csv(ROOT/'attribute_validation.csv')
    ab=attributes.groupby('attribute_basis').unobserved_absrel_pct.mean()
    table_text.append(('Attribute bases: validation diagnostic only',ab.to_string()))
    fig,ax=plt.subplots(figsize=(8,4));ab.plot.bar(ax=ax)
    ax.set_ylabel('Validation unobserved AbsRel (%)');ax.set_xlabel('Basis source (equal dimensions)')
    fig.tight_layout();fig.savefig(REPORT/'attribute_ablation.png',dpi=180);plt.close(fig)
if (ROOT/'attribute_basis_overlap.csv').exists():
    ov=pd.read_csv(ROOT/'attribute_basis_overlap.csv')
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    for ax,field in zip(axs,['left_overlap','right_overlap']):
        matrix=ov.pivot(index='attribute_a',columns='attribute_b',values=field)
        im=ax.imshow(matrix.values,vmin=0,vmax=1,cmap='viridis')
        ax.set_xticks(range(len(matrix)),matrix.columns,rotation=45,ha='right')
        ax.set_yticks(range(len(matrix)),matrix.index);ax.set_title(field)
    fig.colorbar(im,ax=axs.tolist(),label='Normalized projection overlap')
    fig.savefig(REPORT/'attribute_overlap.png',dpi=180,bbox_inches='tight');plt.close(fig)

status=json.loads((ROOT/'stage_status.json').read_text()) if (ROOT/'stage_status.json').exists() else {}
description='''# DA4022 final evaluation: measured results

## Objective
Investigate whether fixed bases derived from controlled 3D variations can guide sparse-depth test-time adaptation, and whether a small residual LoRA restores flexibility.

## Data and separation
This is a small custom ScanNet single-frame adaptation study using the official CAPA sample tensors. Donor sequences: scene0707 and scene0766. Validation: scene0771. Test: scene0777 and scene0806. Two frames per sequence are used. No sequence crosses splits. The original sample conditioning is replaced by explicitly seeded random point measurements from valid dense GT. This is not the original CAPA sequence-level protocol or a standard zero-shot benchmark result.

## Controlled synthetic attributes
A simple analytic room/box/sphere renderer generates geometry, camera, texture, and illumination interventions. Within each adapter task, other scene parameters remain fixed and only the named attribute varies; one adapter is shared across the three variants. Two nuisance base scenes are used per attribute. This is low-diversity procedural data, not a reproduction of MegaSynth. Synthetic depth is known from the renderer. Test RGB/GT do not enter basis extraction.

## Methods
Frozen VGGT + sparse-depth alignment; ordinary image-encoder QKV LoRA rank 4; rank-1 LoRA as a near-budget comparator for the hybrid; random fixed bases; pooled synthetic attribute bases; real-donor bases; and synthetic bases + rank-1 residual LoRA. Main learned bases use a rank-d robust shared update mean, approximated by five p=1 IRLS iterations with low-rank QR/SVD factors. Orthonormal singular bases are retained for coefficient optimization. An additional learned_energy method pools Frobenius-normalized uncentered update energy as an extraction ablation. This is a small independent implementation inspired by the shared-update objective, not a full reproduction of the original paper's data and training pipeline. Four-dimensional left/right bases and a trainable 4x4 matrix per layer are used. Combined and individual attribute bases have equal dimensions, but combined bases use more donor tasks. LoRA and residual scaling are 2. All weight decay is zero. The coefficient and residual learning rates may differ; the exact choices are recorded.

## Selection, masks, and timing
Method learning rates are selected using final unobserved-region error on validation frames at a common 30-update budget, then frozen. Main evaluation uses up to 100 updates. GT is used for metrics and generating the declared sparse measurement simulation; no other dense test supervision enters gradients or alignment. Noise multiplies a seeded subset of observed values by uniform[0.2,2.0] factors. All methods share conditioning and corruption for an image/condition. Invalid predictions cause failed runs rather than exclusion. Nonpositive predictions remain in error metrics and their fractions are recorded. No test-best checkpoint is used for final scores.

Timings include synchronized forward/alignment/backward/optimizer work and exclude metrics, file saving, preprocessing and model loading. These are instrumented pilot timings with checkpointing and warm-cache effects, not deployment latency. Offline donor costs are available in the raw measurements. The reported time-to-target is the first recorded checkpoint reaching a predeclared 25% improvement over the aligned baseline; it is not time to a global optimum.

## Interpretation and limits
Tables are generated only from complete runs. Check scene/frame/seed coverage before comparing aggregates. Frames/seeds are not independent scenes. Only two real donor scenes, one validation scene, two test scenes and two synthetic base scenes are used: no broad generalization or statistical significance claim is justified. Seeds vary LoRA initialization and random bases; pure fixed-basis coefficient runs have no random initialization. CUDA reproducibility is best-effort, and warnings/versions are recorded. Attribute overlap is descriptive and does not establish semantic disentanglement. The quadratic experiment deliberately supplies an optimum-containing subspace and demonstrates a mechanism rather than a VGGT guarantee. Historical mid-evaluation numbers are not mixed into these tables. Any accuracy, convergence or robustness benefit must be supported by the generated results, not assumed.
'''
description+='\n## Stage completion\n\n```json\n'+json.dumps(status,indent=2)+'\n```\n'
description+='\nIncomplete/failed runs: '+str(len(failed))+' (see incomplete_runs.json).\n'
for title,text in table_text:
    description+='\n## '+title+'\n\n```text\n'+text+'\n```\n'
description+='''
## Presentation sequence
1. Proposal: supervision from CAPA, reusable directions from attribute mining.
2. Controlled known-optimum experiment and its misalignment failure.
3. Actual synthetic interventions, donor/validation/test separation, adapter equations.
4. Validation choices and main test accuracy/parameter/convergence tables.
5. Attribute ablation and learned-versus-random findings, including negative outcomes.
6. Residual-adapter, sparsity and outlier results.
7. Limitations, offline cost, failure cases and conclusions limited to the measured evidence.

## Sources
- CAPA code and license: https://github.com/nv-dvl/capa
- VGGT: https://github.com/facebookresearch/vggt
- Attribute-subspace paper: https://arxiv.org/abs/2604.10095
- Original course proposal and historical findings are maintained separately in the local workspace.
'''
(REPORT/'FINAL_REPORT.md').write_text(description,encoding='utf-8')
import html
images=''.join(f'<h2>{html.escape(p.stem)}</h2><img src="{p.name}" style="max-width:100%">' for p in sorted(REPORT.glob('*.png')))
(REPORT/'FINAL_REPORT.html').write_text('<!doctype html><meta charset="utf-8"><title>DA4022 results</title>'
    '<style>body{font:15px system-ui;max-width:1200px;margin:40px auto;padding:20px}pre{white-space:pre-wrap}</style>'
    '<pre>'+html.escape(description)+'</pre>'+images,encoding='utf-8')
journal('report_generated',stage_status=status,completed_measurements=len(ALL),incomplete_runs=len(failed))
print('Reports:',REPORT/'FINAL_REPORT.md',REPORT/'FINAL_REPORT.html')
print('Important: inspect generated tables. No method is assumed to win.')
backup()
